# Speculative Decoding Experiment

**Phases 1-7**: Data preparation -> Baseline -> AR Grid A (0.5B) -> AR Grid B (1.5B) -> Quality/Error Analysis -> Synthesis/Visualization -> DriftDiffuse (parallel diffusion drafter)

| Config | Draft | k | Regime |
|--------|-------|---|--------|
| Baseline x 2 | - | - | det / stoch |
| Grid A x 6 | Qwen2.5-0.5B | {4,8,16} | det / stoch |
| Grid B x 6 | Qwen2.5-1.5B | {4,8,16} | det / stoch |
| DriftDiffuse x N | ~30M masked-diffusion | {8,16} | det |

**Total runs:** 2 baselines + 12 AR speculative + DriftDiffuse sweep = 14 + drift configurations

In [1]:
import os
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

# Ensure src/ is on the path (local execution).
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Force offline-only model usage from local cache.
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Core imports
import torch
import pandas as pd
from tqdm.auto import tqdm

import baseline as baseline_module
import speculative as speculative_module

from config import (
    TARGET_MODEL_ID, DRAFT_MODELS, DATASETS, REGIMES,
    DRAFT_LENGTHS, RESULTS_DIR, STABILITY_DIR, FIGURES_DIR,
    SEED, STABILITY_SEEDS, MANIFESTS_DIR,
 )
from utils import set_seed, get_env_info, write_csv

# Print environment
env = get_env_info()
for k, v in env.items():
    print(f"{k}: {v}")
print("Hugging Face mode: offline-only (local cache)")


def install_notebook_progress_helpers():
    helper_version = 3
    if globals().get("_NOTEBOOK_PROGRESS_INSTALLED_VERSION") == helper_version:
        return

    original_spec_grid = globals().get("_ORIGINAL_RUN_SPECULATIVE_GRID")
    if original_spec_grid is None:
        original_spec_grid = speculative_module.run_speculative_grid
        globals()["_ORIGINAL_RUN_SPECULATIVE_GRID"] = original_spec_grid

    def _short_regime(regime_name: str) -> str:
        return "det" if regime_name == "deterministic" else "stoch"

    def _eta_clock(elapsed_s: float, done_n: int, total_n: int) -> tuple[float, str]:
        if done_n <= 0 or total_n <= done_n:
            return 0.0, "--:--:--"
        remaining_s = max((elapsed_s / done_n) * (total_n - done_n), 0.0)
        eta_ts = datetime.now() + timedelta(seconds=remaining_s)
        return remaining_s, eta_ts.strftime("%H:%M:%S")

    def notebook_run_baseline(
        data: dict[str, list[dict]],
        regime_name: str,
        model=None,
        tokenizer=None,
    ) -> list[dict]:
        regime = REGIMES[regime_name]
        if model is None or tokenizer is None:
            model, tokenizer = baseline_module.load_target_model()

        results = []
        total = sum(len(v) for v in data.values())
        done = 0
        label = f"base {_short_regime(regime_name)}"
        bar = tqdm(
            total=total,
            desc=label,
            dynamic_ncols=True,
            leave=True,
            mininterval=0.5,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
        started_at = time.perf_counter()

        try:
            for task_name, samples in data.items():
                max_new_tokens = DATASETS[task_name]["max_new_tokens"]
                task_total = len(samples)
                for task_idx, sample in enumerate(samples, start=1):
                    bar.set_postfix_str(f"task={task_name} {task_idx}/{task_total}")
                    set_seed(SEED)

                    sample_started_at = time.perf_counter()
                    out = baseline_module.run_baseline_sample(
                        model,
                        tokenizer,
                        sample["prompt"],
                        max_new_tokens,
                        regime.temperature,
                        regime.top_p,
                    )
                    sample_elapsed = time.perf_counter() - sample_started_at

                    row = {
                        "sample_id": sample["sample_id"],
                        "task": task_name,
                        "regime": regime_name,
                        **out,
                    }
                    results.append(row)

                    done += 1
                    elapsed = time.perf_counter() - started_at
                    avg_sample_s = elapsed / done if done else 0.0
                    remaining_s, eta_clock = _eta_clock(elapsed, done, total)
                    bar.update(1)
                    bar.set_postfix_str(
                        f"task={task_name} {task_idx}/{task_total} | last={sample_elapsed:.1f}s | avg={avg_sample_s:.1f}s | rem={remaining_s/60:.1f}m | eta={eta_clock}"
                    )
        finally:
            bar.close()

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        csv_path = RESULTS_DIR / f"baseline_{regime_name}.csv"
        write_csv(csv_path, results)
        print(f"Saved -> {csv_path}")
        return results

    def notebook_run_speculative_grid(
        data: dict[str, list[dict]],
        draft_label: str,
        k: int,
        regime_name: str,
        target_model=None,
        target_tokenizer=None,
        draft_model=None,
        draft_tokenizer=None,
        seed: int = SEED,
        show_realtime_progress: bool = False,
        target_device_label: str | None = None,
        draft_device_label: str | None = None,
        **kwargs,
    ) -> list[dict]:
        regime = REGIMES[regime_name]
        if target_model is None:
            target_model, target_tokenizer = baseline_module.load_target_model()
        if draft_model is None:
            draft_model, draft_tokenizer = speculative_module.load_draft_model(draft_label)

        # Keep compatibility with the original dual-progress implementation.
        if show_realtime_progress:
            return original_spec_grid(
                data,
                draft_label,
                k,
                regime_name,
                target_model,
                target_tokenizer,
                draft_model,
                draft_tokenizer,
                seed=seed,
                show_realtime_progress=True,
                target_device_label=target_device_label,
                draft_device_label=draft_device_label,
                **kwargs,
            )

        results = []
        total = sum(len(v) for v in data.values())
        done = 0
        regime_short = _short_regime(regime_name)
        label = f"spec {draft_label} k={k} {regime_short}"
        bar = tqdm(
            total=total,
            desc=label,
            dynamic_ncols=True,
            leave=True,
            mininterval=0.5,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
        started_at = time.perf_counter()

        try:
            for task_name, samples in data.items():
                max_new_tokens = DATASETS[task_name]["max_new_tokens"]
                task_total = len(samples)
                for task_idx, sample in enumerate(samples, start=1):
                    bar.set_postfix_str(f"task={task_name} {task_idx}/{task_total}")
                    set_seed(seed)

                    sample_started_at = time.perf_counter()
                    out = speculative_module.speculative_decode_sample(
                        target_model,
                        draft_model,
                        target_tokenizer,
                        sample["prompt"],
                        max_new_tokens,
                        k,
                        regime.temperature,
                        regime.top_p,
                    )
                    sample_elapsed = time.perf_counter() - sample_started_at

                    row = {
                        "sample_id": sample["sample_id"],
                        "task": task_name,
                        "draft": draft_label,
                        "k": k,
                        "regime": regime_name,
                        "seed": seed,
                        **{key: val for key, val in out.items() if key != "verify_log"},
                    }
                    results.append(row)

                    done += 1
                    elapsed = time.perf_counter() - started_at
                    avg_sample_s = elapsed / done if done else 0.0
                    remaining_s, eta_clock = _eta_clock(elapsed, done, total)
                    bar.update(1)
                    bar.set_postfix_str(
                        f"task={task_name} {task_idx}/{task_total} | last={sample_elapsed:.1f}s | avg={avg_sample_s:.1f}s | rem={remaining_s/60:.1f}m | eta={eta_clock}"
                    )
        finally:
            bar.close()

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        csv_path = RESULTS_DIR / f"spec_{draft_label}_k{k}_{regime_short}.csv"
        write_csv(csv_path, results)
        print(f"Saved -> {csv_path}")
        return results

    globals()["notebook_run_baseline"] = notebook_run_baseline
    globals()["notebook_run_speculative_grid"] = notebook_run_speculative_grid
    globals()["run_baseline"] = notebook_run_baseline
    globals()["run_speculative_grid"] = notebook_run_speculative_grid

    baseline_module.run_baseline = notebook_run_baseline
    speculative_module.run_speculative_grid = notebook_run_speculative_grid

    globals()["_NOTEBOOK_PROGRESS_INSTALLED"] = True
    globals()["_NOTEBOOK_PROGRESS_INSTALLED_VERSION"] = helper_version
    print("Notebook progress helpers installed")


install_notebook_progress_helpers()



python: 
torch: 2.11.0+cu128
cuda_available: True
cuda_version: 12.8
gpu_name: NVIDIA GeForce RTX 4090
device: cuda
transformers: 5.5.4
Hugging Face mode: offline-only (local cache)
Notebook progress helpers installed


## Phase 1 — Data Preparation & Reproducibility Lock

Load GSM8K (300), MMLU (5×100), CNN/DailyMail (200) with `seed=42`, freeze manifests.

In [2]:
# Hugging Face offline-only mode (force local cache usage).
import os

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("Offline-only mode enabled: local Hugging Face cache only.")


Offline-only mode enabled: local Hugging Face cache only.


In [3]:
from data_loader import load_all_datasets, freeze_manifests, save_full_data, load_from_manifests

# Check if manifests already exist (skip download if so)
manifests_exist = all(
    (MANIFESTS_DIR / f"{t}_data.json").exists()
    for t in ["gsm8k", "mmlu", "cnndm"]
)

if manifests_exist:
    print("Manifests found — loading from disk (no re-download)")
    data = load_from_manifests()
else:
    print("Downloading and sampling datasets…")
    data = load_all_datasets()
    freeze_manifests(data)
    save_full_data(data)

# Quick sanity check
for task, samples in data.items():
    print(f"  {task}: {len(samples)} samples, first id: {samples[0]['sample_id']}")

Manifests found — loading from disk (no re-download)
  [gsm8k] loaded 300 samples from manifest
  [mmlu] loaded 500 samples from manifest
  [cnndm] loaded 200 samples from manifest
  gsm8k: 300 samples, first id: gsm8k_2
  mmlu: 500 samples, first id: mmlu_abstract_algebra_0
  cnndm: 200 samples, first id: cnndm_3


### Verify Tokenizer Compatibility

In [4]:
from data_loader import verify_tokenizer_compatibility
import os

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

compatible = verify_tokenizer_compatibility()
assert compatible, "Tokenizer mismatch; cannot proceed with speculative decoding."


  Qwen/Qwen2.5-3B-Instruct: vocab_size = 151643
  Qwen/Qwen2.5-0.5B-Instruct: vocab_size = 151643
  Qwen/Qwen2.5-1.5B-Instruct: vocab_size = 151643
  ✓ All tokenizers share vocab_size = 151643


## Phase 2 — Baseline Benchmarking

Run Qwen2.5-3B-Instruct autoregressive decoding on all 1,000 samples in both regimes.

In [5]:
import os
import sys
import importlib
from pathlib import Path

# Ensure src/ is on the path (local execution).
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

cwd = Path.cwd().resolve()
def _find_project_root(start):
    for c in [start, *start.parents]:
        if (c / "src" / "baseline.py").exists():
            return c
    return start

project_root = _find_project_root(cwd)
SRC_DIR = str(project_root / "src")
if not (project_root / "src" / "baseline.py").exists():
    raise ModuleNotFoundError(
        f"Could not find src/baseline.py from cwd={cwd}."
    )
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Force-refresh modules to avoid stale target-model config in long-lived kernels.
import config as config_module
import baseline as baseline_module
import runtime as runtime_module
config_module = importlib.reload(config_module)
baseline_module = importlib.reload(baseline_module)
runtime_module = importlib.reload(runtime_module)

from runtime import bootstrap_notebook, ensure_data, ensure_target_model
from baseline import run_baseline
from config import TARGET_MODEL_ID

bootstrap_notebook()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

data = ensure_data(globals())
target_model, target_tokenizer = ensure_target_model(globals())

loaded_target = str(getattr(target_model, "name_or_path", ""))
print(f"Configured target model: {TARGET_MODEL_ID}")
print(f"Loaded target model: {loaded_target}")
assert TARGET_MODEL_ID.endswith("Qwen2.5-3B-Instruct"), f"Unexpected TARGET_MODEL_ID: {TARGET_MODEL_ID}"
assert loaded_target == TARGET_MODEL_ID, f"Target mismatch: loaded={loaded_target}, expected={TARGET_MODEL_ID}"

if "baseline_det" not in globals() and (RESULTS_DIR / "baseline_deterministic.csv").exists():
    baseline_det = pd.read_csv(RESULTS_DIR / "baseline_deterministic.csv").to_dict(orient="records")
if "baseline_stoch" not in globals() and (RESULTS_DIR / "baseline_stochastic.csv").exists():
    baseline_stoch = pd.read_csv(RESULTS_DIR / "baseline_stochastic.csv").to_dict(orient="records")


Loading target model: Qwen/Qwen2.5-3B-Instruct (quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Configured target model: Qwen/Qwen2.5-3B-Instruct
Loaded target model: Qwen/Qwen2.5-3B-Instruct


In [6]:
# --- Deterministic baseline ---
print("=" * 60)
print("Baseline: DETERMINISTIC regime")
print("=" * 60)
if globals().get("baseline_det"):
    print("Using existing deterministic baseline from memory/disk")
else:
    baseline_det = notebook_run_baseline(data, "deterministic", target_model, target_tokenizer)



Baseline: DETERMINISTIC regime
Using existing deterministic baseline from memory/disk


In [7]:
# --- Stochastic baseline ---
print("=" * 60)
print("Baseline: STOCHASTIC regime")
print("=" * 60)
if globals().get("baseline_stoch"):
    print("Using existing stochastic baseline from memory/disk")
else:
    baseline_stoch = notebook_run_baseline(data, "stochastic", target_model, target_tokenizer)



Baseline: STOCHASTIC regime
Using existing stochastic baseline from memory/disk


In [8]:
# --- Evaluate baseline quality ---
from evaluate import evaluate_results

print("\nBaseline quality — Deterministic:")
base_quality_det = evaluate_results(baseline_det, data)

print("\nBaseline quality — Stochastic:")
base_quality_stoch = evaluate_results(baseline_stoch, data)


Baseline quality — Deterministic:
  gsm8k (exact_match): 59.00%  (n=300)
  mmlu (letter_match): 56.60%  (n=500)
  cnndm (rouge_l): 20.90%  (n=200)

Baseline quality — Stochastic:
  gsm8k (exact_match): 56.33%  (n=300)
  mmlu (letter_match): 56.80%  (n=500)
  cnndm (rouge_l): 20.38%  (n=200)


In [9]:
# --- Quick latency summary ---
from metrics import compute_latency_metrics

print("\nBaseline latency — Deterministic:")
base_lat_det = compute_latency_metrics(baseline_det)
for k, v in base_lat_det.items():
    print(f"  {k}: {v}")

print("\nBaseline latency — Stochastic:")
base_lat_stoch = compute_latency_metrics(baseline_stoch)
for k, v in base_lat_stoch.items():
    print(f"  {k}: {v}")


Baseline latency — Deterministic:
  T_mean_s: 11.4139
  T_median_s: 7.4798
  R_tok_mean: 11.03
  TTFT_mean_ms: 233.53
  TPOT_mean_ms: 89.26
  total_tokens: 124800
  n_samples: 1000

Baseline latency — Stochastic:
  T_mean_s: 11.6492
  T_median_s: 7.2956
  R_tok_mean: 10.83
  TTFT_mean_ms: 232.94
  TPOT_mean_ms: 91.11
  total_tokens: 124800
  n_samples: 1000


## Phase 2.5 — Dual-GPU 3B/3B Speculative Pilot (First 200 Samples)

Load target 3B on `cuda:0` and a second 3B as draft on `cuda:1`, then run speculative decoding on the first 200 samples (`k=4`) for both regimes. Output CSV format matches existing speculative result files for unified analysis.

In [10]:
import os
import sys
from pathlib import Path

import torch

# Ensure src/ is importable even when this cell is run independently.
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src" / "runtime.py").exists()), cwd)
SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from runtime import bootstrap_notebook, ensure_data, ensure_dual_3b_results
from config import TARGET_MODEL_ID, RESULTS_DIR

bootstrap_notebook()

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Dual-GPU run requires at least 2 CUDA devices: cuda:0 and cuda:1")

data = ensure_data(globals())

# Show exactly how many rows are taken per task from the global first-200 slice.
remaining = 200
subset_breakdown = {}
for task_name, samples in data.items():
    if remaining <= 0:
        break
    take_n = min(len(samples), remaining)
    if take_n > 0:
        subset_breakdown[task_name] = take_n
        remaining -= take_n

print(f"Target model id: {TARGET_MODEL_ID}")
print(f"CUDA devices: {torch.cuda.device_count()}")
print(f"Subset breakdown (first 200 total): {subset_breakdown}")
print("Live compare: showing two progress bars (draft on cuda:1, target verify on cuda:0) with separate ETA")

spec_results_3b_dual = ensure_dual_3b_results(
    globals(),
    k=4,
    max_samples=200,
    target_device="cuda:0",
    draft_device="cuda:1",
    regimes=("deterministic", "stochastic"),
    draft_label="3B_dual",
    show_realtime_progress=True,
    force_rerun=True,
 )

print("Saved CSV files:")
for regime_name in ("deterministic", "stochastic"):
    short = "det" if regime_name == "deterministic" else "stoch"
    print(f"  - {RESULTS_DIR / f'spec_3B_dual_k4_{short}.csv'}")

print(f"Completed dual-3B configs: {list(spec_results_3b_dual.keys())}")

Target model id: Qwen/Qwen2.5-3B-Instruct
CUDA devices: 2
Subset breakdown (first 200 total): {'gsm8k': 200}
Live compare: showing two progress bars (draft on cuda:1, target verify on cuda:0) with separate ETA
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:1, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

draft 3B_dual cuda:1:   0%|          | 0/200 [00:00<?, ?it/s]

target verify cuda:0:   0%|          | 0/200 [00:00<?, ?it/s]

  Saved -> C:\Working\speculative-decoding-main\results\spec_3B_dual_k4_det.csv
  Verify logs -> C:\Working\speculative-decoding-main\results\verify_logs\spec_3B_dual_k4_det_seed42.json


draft 3B_dual cuda:1:   0%|          | 0/200 [00:00<?, ?it/s]

target verify cuda:0:   0%|          | 0/200 [00:00<?, ?it/s]

  Saved -> C:\Working\speculative-decoding-main\results\spec_3B_dual_k4_stoch.csv
  Verify logs -> C:\Working\speculative-decoding-main\results\verify_logs\spec_3B_dual_k4_stoch_seed42.json
Saved CSV files:
  - C:\Working\speculative-decoding-main\results\spec_3B_dual_k4_det.csv
  - C:\Working\speculative-decoding-main\results\spec_3B_dual_k4_stoch.csv
Completed dual-3B configs: ['3B_dual_k4_deterministic', '3B_dual_k4_stochastic']


In [2]:
# --- Quick dual-3B independent smoke test (10 English samples, <=5 min target) ---
import os
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

# 1) Self-contained environment setup (no dependency on previous cells).
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src" / "runtime.py").exists()), cwd)
SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from config import TARGET_MODEL_ID, TARGET_QUANT, DRAFT_QUANT, RESULTS_DIR
from utils import set_seed
from baseline import run_baseline_sample
from speculative import load_model_on_device, speculative_decode_sample

# 2) Runtime guardrails for quick run.
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("This quick dual-3B test requires 2 CUDA devices: cuda:0 and cuda:1")

# Keep this intentionally small to finish quickly.
QUICK_SEED = 42
QUICK_K = 4
QUICK_MAX_NEW_TOKENS = 32
QUICK_TEMP = 0.0
QUICK_TOP_P = 1.0
set_seed(QUICK_SEED)

# 3) 10 short English samples (standalone, no dataset dependency).
quick_samples = [
    {"sample_id": "quick_en_001", "task": "quick_en", "prompt": "Explain in two sentences why the sky appears blue."},
    {"sample_id": "quick_en_002", "task": "quick_en", "prompt": "Give three practical tips to improve sleep quality."},
    {"sample_id": "quick_en_003", "task": "quick_en", "prompt": "Summarize what machine learning is in plain English."},
    {"sample_id": "quick_en_004", "task": "quick_en", "prompt": "Write a short comparison between Python and C++."},
    {"sample_id": "quick_en_005", "task": "quick_en", "prompt": "List five healthy breakfast ideas with brief descriptions."},
    {"sample_id": "quick_en_006", "task": "quick_en", "prompt": "Describe how photosynthesis works for a middle-school student."},
    {"sample_id": "quick_en_007", "task": "quick_en", "prompt": "Give a concise step-by-step plan to prepare for a job interview."},
    {"sample_id": "quick_en_008", "task": "quick_en", "prompt": "Explain the difference between weather and climate."},
    {"sample_id": "quick_en_009", "task": "quick_en", "prompt": "Write a short paragraph on the importance of data privacy."},
    {"sample_id": "quick_en_010", "task": "quick_en", "prompt": "Provide a simple 3-day beginner workout routine."},
]

# 4) Load models explicitly on dual GPUs.
print(f"Loading target model on cuda:0 -> {TARGET_MODEL_ID}")
target_model, target_tokenizer = load_model_on_device(
    TARGET_MODEL_ID,
    "cuda:0",
    TARGET_QUANT,
 )

print(f"Loading 3B draft model on cuda:1 -> {TARGET_MODEL_ID}")
draft_model, draft_tokenizer = load_model_on_device(
    TARGET_MODEL_ID,
    "cuda:1",
    DRAFT_QUANT,
 )

print(f"Target device: {target_model.device}")
print(f"Draft device: {next(draft_model.parameters()).device}")
print(f"Quick run config: n={len(quick_samples)}, k={QUICK_K}, max_new_tokens={QUICK_MAX_NEW_TOKENS}, regime=deterministic")

# 5) Baseline run (target-only on cuda:0) with progress.
baseline_rows = []
base_started = time.perf_counter()
base_bar = tqdm(total=len(quick_samples), desc="quick baseline cuda:0", dynamic_ncols=True, leave=True)
for idx, sample in enumerate(quick_samples, start=1):
    set_seed(QUICK_SEED)
    out = run_baseline_sample(
        target_model,
        target_tokenizer,
        sample["prompt"],
        QUICK_MAX_NEW_TOKENS,
        QUICK_TEMP,
        QUICK_TOP_P,
    )
    baseline_rows.append({
        "sample_id": sample["sample_id"],
        "task": sample["task"],
        "mode": "baseline",
        "k": 0,
        "regime": "deterministic",
        "seed": QUICK_SEED,
        **out,
    })
    base_bar.update(1)
    base_bar.set_postfix_str(f"{idx}/{len(quick_samples)}")
base_bar.close()
base_elapsed = time.perf_counter() - base_started

# 6) Speculative run (target on cuda:0, draft 3B on cuda:1) with progress.
spec_rows = []
spec_started = time.perf_counter()
spec_bar = tqdm(total=len(quick_samples), desc="quick spec 3B/3B dual", dynamic_ncols=True, leave=True)
for idx, sample in enumerate(quick_samples, start=1):
    set_seed(QUICK_SEED)
    out = speculative_decode_sample(
        target_model,
        draft_model,
        target_tokenizer,
        sample["prompt"],
        QUICK_MAX_NEW_TOKENS,
        QUICK_K,
        QUICK_TEMP,
        QUICK_TOP_P,
        return_timing_breakdown=True,
        adaptive_k=True,
    )
    spec_rows.append({
        "sample_id": sample["sample_id"],
        "task": sample["task"],
        "mode": "speculative",
        "draft": "3B_dual_quick",
        "k": QUICK_K,
        "regime": "deterministic",
        "seed": QUICK_SEED,
        **{k: v for k, v in out.items() if k != "verify_log"},
    })
    spec_bar.update(1)
    spec_bar.set_postfix_str(f"{idx}/{len(quick_samples)}")
spec_bar.close()
spec_elapsed = time.perf_counter() - spec_started

# 7) Save outputs.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
baseline_csv = RESULTS_DIR / "quick_dual3b_baseline_10samples.csv"
spec_csv = RESULTS_DIR / "quick_dual3b_spec_10samples.csv"
summary_csv = RESULTS_DIR / "quick_dual3b_summary_10samples.csv"

pd.DataFrame(baseline_rows).to_csv(baseline_csv, index=False)
pd.DataFrame(spec_rows).to_csv(spec_csv, index=False)

# 8) Pairwise summary.
base_by_id = {r["sample_id"]: r for r in baseline_rows}
paired = []
for r in spec_rows:
    b = base_by_id.get(r["sample_id"]);
    if not b:
        continue
    b_lat = float(b.get("latency_s", 0.0))
    s_lat = float(r.get("latency_s", 0.0))
    paired.append({
        "sample_id": r["sample_id"],
        "baseline_latency_s": b_lat,
        "spec_latency_s": s_lat,
        "speedup_baseline_over_spec": (b_lat / s_lat) if s_lat > 0 else 0.0,
        "baseline_tokens_per_sec": float(b.get("tokens_per_sec", 0.0)),
        "spec_tokens_per_sec": float(r.get("tokens_per_sec", 0.0)),
        "alpha": float(r.get("alpha", 0.0)),
        "B_eff": float(r.get("B_eff", 0.0)),
        "draft_elapsed_s": float(r.get("draft_elapsed_s", 0.0)),
        "verify_elapsed_s": float(r.get("verify_elapsed_s", 0.0)),
    })

df_pair = pd.DataFrame(paired)
if not df_pair.empty:
    df_pair.to_csv(summary_csv, index=False)

print("=" * 70)
print("Quick dual-3B smoke test done")
print("=" * 70)
print(f"Baseline wall time:    {base_elapsed:.2f}s")
print(f"Speculative wall time: {spec_elapsed:.2f}s")
print(f"Saved baseline CSV -> {baseline_csv}")
print(f"Saved spec CSV     -> {spec_csv}")
if not df_pair.empty:
    mean_speedup = float(df_pair["speedup_baseline_over_spec"].mean())
    mean_alpha = float(df_pair["alpha"].mean())
    mean_draft = float(df_pair["draft_elapsed_s"].mean())
    mean_verify = float(df_pair["verify_elapsed_s"].mean())
    print(f"Saved pair summary -> {summary_csv}")
    print(f"Mean speedup (baseline/spec): {mean_speedup:.4f}x")
    print(f"Mean alpha: {mean_alpha:.4f}")
    print(f"Mean draft time: {mean_draft:.4f}s | mean verify time: {mean_verify:.4f}s")
else:
    print("No paired rows found in quick summary.")

if (base_elapsed + spec_elapsed) > 300:
    print("WARNING: Total runtime exceeded 5 minutes. You can reduce QUICK_MAX_NEW_TOKENS to 24 or 16.")

Loading target model on cuda:0 -> Qwen/Qwen2.5-3B-Instruct
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading 3B draft model on cuda:1 -> Qwen/Qwen2.5-3B-Instruct
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:1, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Target device: cuda:0
Draft device: cuda:1
Quick run config: n=10, k=4, max_new_tokens=32, regime=deterministic


quick baseline cuda:0:   0%|          | 0/10 [00:00<?, ?it/s]

quick spec 3B/3B dual:   0%|          | 0/10 [00:00<?, ?it/s]

Quick dual-3B smoke test done
Baseline wall time:    7.40s
Speculative wall time: 42.62s
Saved baseline CSV -> C:\Working\speculative-decoding-main\results\quick_dual3b_baseline_10samples.csv
Saved spec CSV     -> C:\Working\speculative-decoding-main\results\quick_dual3b_spec_10samples.csv
Saved pair summary -> C:\Working\speculative-decoding-main\results\quick_dual3b_summary_10samples.csv
Mean speedup (baseline/spec): 0.1743x
Mean alpha: 0.4607
Mean draft time: 2.7267s | mean verify time: 0.3325s


In [1]:
# --- Quick acceleration-oriented quantization A/B test (independent, 10 samples) ---
import os
import sys
import time
import gc
import traceback
from pathlib import Path
from datetime import datetime

# Robustness on Windows: avoid noisy/non-UTF8 banners from optional backends.
os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("BITSANDBYTES_NOWELCOME", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import pandas as pd
import torch
from tqdm.auto import tqdm

# Self-contained environment setup.
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src" / "runtime.py").exists()), cwd)
SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from config import TARGET_MODEL_ID, DRAFT_MODELS, RESULTS_DIR
from baseline import run_baseline_sample
from speculative import load_model_on_device, speculative_decode_sample
from quantization import get_quant_kwargs

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this quick test")

# Tunables
QUICK_K = 4
QUICK_MAX_NEW_TOKENS = 12
QUICK_TEMP = 0.0
QUICK_TOP_P = 1.0
QUICK_ADAPTIVE_K = False
QUICK_DRAFT_PROFILE = "0.5B"  # "0.5B" or "3B"
QUICK_COLOCATE_DRAFT_WITH_TARGET = True

# Quantization profile matrix (same prompts, same decode config).
QUICK_QUANT_PROFILES = [
    {"name": "fp16_target_fp16_draft", "target_quant": "fp16", "draft_quant": "fp16"},
    {"name": "int8_target_fp16_draft", "target_quant": "int8", "draft_quant": "fp16"},
    # {"name": "int8_target_int8_draft", "target_quant": "int8", "draft_quant": "int8"},
]

if QUICK_DRAFT_PROFILE == "3B":
    quick_draft_model_id = TARGET_MODEL_ID
    quick_draft_label = "3B_quick"
else:
    quick_draft_model_id = DRAFT_MODELS["0.5B"]
    quick_draft_label = "0.5B_quick"

target_device = "cuda:0"
if QUICK_COLOCATE_DRAFT_WITH_TARGET or torch.cuda.device_count() < 2:
    draft_device = "cuda:0"
else:
    draft_device = "cuda:1"

if quick_draft_model_id == TARGET_MODEL_ID:
    print("WARNING: draft uses the same 3B model as target; slowdown is expected.")

quick_samples = [
    {"sample_id": "fast_en_001", "task": "quick_en", "prompt": "Explain in one sentence why leaves are green."},
    {"sample_id": "fast_en_002", "task": "quick_en", "prompt": "Give two tips for improving concentration while studying."},
    {"sample_id": "fast_en_003", "task": "quick_en", "prompt": "Define cloud computing in plain language."},
    {"sample_id": "fast_en_004", "task": "quick_en", "prompt": "Name three benefits of regular exercise."},
    {"sample_id": "fast_en_005", "task": "quick_en", "prompt": "Briefly explain what an API is."},
    {"sample_id": "fast_en_006", "task": "quick_en", "prompt": "Compare supervised and unsupervised learning in two lines."},
    {"sample_id": "fast_en_007", "task": "quick_en", "prompt": "What is overfitting in machine learning?"},
    {"sample_id": "fast_en_008", "task": "quick_en", "prompt": "List two ways to reduce energy consumption at home."},
    {"sample_id": "fast_en_009", "task": "quick_en", "prompt": "Explain the main idea of version control."},
    {"sample_id": "fast_en_010", "task": "quick_en", "prompt": "Write one short tip for clear technical writing."},
]

def _resolved_quant(mode):
    _, resolved = get_quant_kwargs(mode)
    return resolved

def _cuda_mem_mb(device: str) -> dict:
    if not torch.cuda.is_available() or not str(device).startswith("cuda"):
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0, "max_reserved_mb": 0.0}
    idx = int(str(device).split(":")[1])
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
        "max_reserved_mb": torch.cuda.max_memory_reserved(idx) / (1024 ** 2),
    }

def _safe_dtype_of_model(model):
    try:
        return str(next(model.parameters()).dtype)
    except Exception:
        return "unknown"

def _safe_cleanup_models(target_model=None, draft_model=None):
    try:
        if draft_model is not None:
            del draft_model
    except Exception:
        pass
    try:
        if target_model is not None:
            del target_model
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _profile_header(pname, tq, dq):
    print("\n" + "=" * 96)
    print(f"Profile: {pname}")
    print(
        f"Target quant request/resolved: {tq}/{_resolved_quant(tq)} | "
        f"Draft quant request/resolved: {dq}/{_resolved_quant(dq)}"
    )
    print(
        f"Config: n={len(quick_samples)} k={QUICK_K} max_new_tokens={QUICK_MAX_NEW_TOKENS} "
        f"draft_profile={QUICK_DRAFT_PROFILE} colocate={QUICK_COLOCATE_DRAFT_WITH_TARGET} adaptive_k={QUICK_ADAPTIVE_K}"
    )

def run_profile(profile: dict) -> tuple[dict, pd.DataFrame]:
    pname = profile["name"]
    tq = profile["target_quant"]
    dq = profile["draft_quant"]

    target_model = None
    draft_model = None

    debug_meta = {
        "profile": pname,
        "target_quant_req": tq,
        "draft_quant_req": dq,
        "target_quant_resolved": _resolved_quant(tq),
        "draft_quant_resolved": _resolved_quant(dq),
        "target_load_s": float("nan"),
        "draft_load_s": float("nan"),
        "target_dtype": "",
        "draft_dtype": "",
        "target_device": target_device,
        "draft_device": draft_device,
        "cuda_count": int(torch.cuda.device_count()),
        "exception_type": "",
        "exception_message": "",
        "traceback": "",
    }

    df_pairs = pd.DataFrame()

    try:
        _profile_header(pname, tq, dq)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        print(f"Loading target on {target_device} -> {TARGET_MODEL_ID}")
        t_load_0 = time.perf_counter()
        target_model, target_tokenizer = load_model_on_device(
            TARGET_MODEL_ID,
            target_device,
            tq,
        )
        debug_meta["target_load_s"] = time.perf_counter() - t_load_0
        debug_meta["target_dtype"] = _safe_dtype_of_model(target_model)

        print(f"Loading draft on {draft_device} -> {quick_draft_model_id}")
        d_load_0 = time.perf_counter()
        draft_model, _draft_tokenizer = load_model_on_device(
            quick_draft_model_id,
            draft_device,
            dq,
        )
        debug_meta["draft_load_s"] = time.perf_counter() - d_load_0
        debug_meta["draft_dtype"] = _safe_dtype_of_model(draft_model)

        m_target_after_load = _cuda_mem_mb(target_device)
        m_draft_after_load = _cuda_mem_mb(draft_device)

        print("Model load diagnostics:")
        print(
            f"  target_load_s={debug_meta['target_load_s']:.2f}s dtype={debug_meta['target_dtype']} "
            f"alloc/reserved={m_target_after_load['alloc_mb']:.0f}/{m_target_after_load['reserved_mb']:.0f}MB"
        )
        print(
            f"  draft_load_s={debug_meta['draft_load_s']:.2f}s dtype={debug_meta['draft_dtype']} "
            f"alloc/reserved={m_draft_after_load['alloc_mb']:.0f}/{m_draft_after_load['reserved_mb']:.0f}MB"
        )

        baseline_rows = []
        t0 = time.perf_counter()
        bar_base = tqdm(total=len(quick_samples), desc=f"base {pname}", dynamic_ncols=True, leave=True)
        for i, sample in enumerate(quick_samples, start=1):
            out = run_baseline_sample(
                target_model,
                target_tokenizer,
                sample["prompt"],
                QUICK_MAX_NEW_TOKENS,
                QUICK_TEMP,
                QUICK_TOP_P,
            )
            baseline_rows.append({
                "sample_id": sample["sample_id"],
                "task": sample["task"],
                "mode": "baseline",
                "k": 0,
                "regime": "deterministic",
                **out,
            })
            bar_base.update(1)
            bar_base.set_postfix_str(f"{i}/{len(quick_samples)}")
        bar_base.close()
        base_elapsed = time.perf_counter() - t0

        spec_rows = []
        t1 = time.perf_counter()
        bar_spec = tqdm(total=len(quick_samples), desc=f"spec {pname}", dynamic_ncols=True, leave=True)
        for i, sample in enumerate(quick_samples, start=1):
            out = speculative_decode_sample(
                target_model,
                draft_model,
                target_tokenizer,
                sample["prompt"],
                QUICK_MAX_NEW_TOKENS,
                QUICK_K,
                QUICK_TEMP,
                QUICK_TOP_P,
                return_timing_breakdown=True,
                adaptive_k=QUICK_ADAPTIVE_K,
            )
            spec_rows.append({
                "sample_id": sample["sample_id"],
                "task": sample["task"],
                "mode": "speculative",
                "draft": quick_draft_label,
                "k": QUICK_K,
                "regime": "deterministic",
                "target_device": target_device,
                "draft_device": draft_device,
                "adaptive_k": QUICK_ADAPTIVE_K,
                "profile": pname,
                **{k: v for k, v in out.items() if k != "verify_log"},
            })
            bar_spec.update(1)
            bar_spec.set_postfix_str(f"{i}/{len(quick_samples)}")
        bar_spec.close()
        spec_elapsed = time.perf_counter() - t1

        m_target_after_run = _cuda_mem_mb(target_device)
        m_draft_after_run = _cuda_mem_mb(draft_device)

        base_by_id = {r["sample_id"]: r for r in baseline_rows}
        pairs = []
        for r in spec_rows:
            b = base_by_id.get(r["sample_id"])
            if not b:
                continue

            b_lat = float(b.get("latency_s", 0.0))
            s_lat = float(r.get("latency_s", 0.0))
            draft_t = float(r.get("draft_elapsed_s", 0.0))
            verify_t = float(r.get("verify_elapsed_s", 0.0))
            other_t = max(s_lat - draft_t - verify_t, 0.0)
            timing_gap = s_lat - (draft_t + verify_t + other_t)
            delta_t = s_lat - b_lat
            spec_denom = max(s_lat, 1e-9)

            pairs.append({
                "sample_id": r["sample_id"],
                "profile": pname,
                "baseline_latency_s": b_lat,
                "spec_latency_s": s_lat,
                "extra_vs_baseline_s": delta_t,
                "is_slower_than_baseline": int(delta_t > 0.0),
                "speedup_baseline_over_spec": (b_lat / s_lat) if s_lat > 0 else 0.0,
                "alpha": float(r.get("alpha", 0.0)),
                "B_eff": float(r.get("B_eff", 0.0)),
                "n_verify_steps": float(r.get("n_verify_steps", 0.0)),
                "draft_elapsed_s": draft_t,
                "verify_elapsed_s": verify_t,
                "other_elapsed_s": other_t,
                "timing_remainder_s": timing_gap,
                "draft_share_of_spec": draft_t / spec_denom,
                "verify_share_of_spec": verify_t / spec_denom,
                "other_share_of_spec": other_t / spec_denom,
            })

        df_pairs = pd.DataFrame(pairs)

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        suffix = f"{quick_draft_label}_{pname}_k{QUICK_K}_tok{QUICK_MAX_NEW_TOKENS}_{draft_device.replace(':','')}_{'adapt' if QUICK_ADAPTIVE_K else 'fixed'}"
        base_csv = RESULTS_DIR / f"quick_fast_baseline_10samples_{suffix}.csv"
        spec_csv = RESULTS_DIR / f"quick_fast_spec_10samples_{suffix}.csv"
        pair_csv = RESULTS_DIR / f"quick_fast_pair_10samples_{suffix}.csv"
        breakdown_csv = RESULTS_DIR / f"quick_fast_breakdown_10samples_{suffix}.csv"
        debug_csv = RESULTS_DIR / f"quick_fast_debug_metrics_10samples_{suffix}.csv"

        pd.DataFrame(baseline_rows).to_csv(base_csv, index=False)
        pd.DataFrame(spec_rows).to_csv(spec_csv, index=False)
        if not df_pairs.empty:
            df_pairs.to_csv(pair_csv, index=False)
            df_pairs.to_csv(breakdown_csv, index=False)

        debug_rows = [{
            **debug_meta,
            "target_alloc_mb_after_load": m_target_after_load["alloc_mb"],
            "target_reserved_mb_after_load": m_target_after_load["reserved_mb"],
            "target_peak_alloc_mb_after_run": m_target_after_run["max_alloc_mb"],
            "target_peak_reserved_mb_after_run": m_target_after_run["max_reserved_mb"],
            "draft_alloc_mb_after_load": m_draft_after_load["alloc_mb"],
            "draft_reserved_mb_after_load": m_draft_after_load["reserved_mb"],
            "draft_peak_alloc_mb_after_run": m_draft_after_run["max_alloc_mb"],
            "draft_peak_reserved_mb_after_run": m_draft_after_run["max_reserved_mb"],
            "base_wall_s": base_elapsed,
            "spec_wall_s": spec_elapsed,
            "n_samples": int(len(df_pairs)),
        }]
        pd.DataFrame(debug_rows).to_csv(debug_csv, index=False)

        total_spec = float(df_pairs["spec_latency_s"].sum()) if not df_pairs.empty else 0.0
        total_draft = float(df_pairs["draft_elapsed_s"].sum()) if not df_pairs.empty else 0.0
        total_verify = float(df_pairs["verify_elapsed_s"].sum()) if not df_pairs.empty else 0.0
        total_other = float(df_pairs["other_elapsed_s"].sum()) if not df_pairs.empty else 0.0
        total_extra = float(df_pairs["extra_vs_baseline_s"].sum()) if not df_pairs.empty else 0.0
        spec_denom = max(total_spec, 1e-9)

        mean_speedup = float(df_pairs["speedup_baseline_over_spec"].mean()) if not df_pairs.empty else 0.0
        mean_alpha = float(df_pairs["alpha"].mean()) if not df_pairs.empty else 0.0
        mean_verify_steps = float(df_pairs["n_verify_steps"].mean()) if not df_pairs.empty else 0.0
        slower_count = int(df_pairs["is_slower_than_baseline"].sum()) if not df_pairs.empty else 0
        mean_timing_remainder = float(df_pairs["timing_remainder_s"].mean()) if not df_pairs.empty else 0.0

        print(f"Baseline wall time:    {base_elapsed:.2f}s")
        print(f"Speculative wall time: {spec_elapsed:.2f}s")
        print(f"Mean speedup (baseline/spec): {mean_speedup:.4f}x")
        print(f"Mean alpha: {mean_alpha:.4f} | mean verify steps: {mean_verify_steps:.2f}")
        print(f"Where speculative runtime is spent: draft={100.0*total_draft/spec_denom:.1f}% verify={100.0*total_verify/spec_denom:.1f}% other={100.0*total_other/spec_denom:.1f}%")
        print(f"Net extra vs baseline (sum): {total_extra:.4f}s | slower samples: {slower_count}/{len(df_pairs)}")
        print(f"Timing remainder mean (spec - draft - verify - other): {mean_timing_remainder:.6f}s")
        print(f"Saved pair CSV -> {pair_csv}")
        print(f"Saved breakdown CSV -> {breakdown_csv}")
        print(f"Saved debug CSV -> {debug_csv}")

        if not df_pairs.empty:
            cols = [
                "sample_id", "extra_vs_baseline_s", "alpha", "n_verify_steps",
                "draft_elapsed_s", "verify_elapsed_s", "other_elapsed_s",
            ]
            print("Top-3 slowest samples:")
            print(df_pairs.sort_values("extra_vs_baseline_s", ascending=False).head(3)[cols].to_string(index=False))

        return {
            "profile": pname,
            "status": "ok",
            "error": "",
            "target_quant_req": tq,
            "draft_quant_req": dq,
            "target_quant_resolved": _resolved_quant(tq),
            "draft_quant_resolved": _resolved_quant(dq),
            "target_dtype": debug_meta["target_dtype"],
            "draft_dtype": debug_meta["draft_dtype"],
            "target_load_s": debug_meta["target_load_s"],
            "draft_load_s": debug_meta["draft_load_s"],
            "base_wall_s": base_elapsed,
            "spec_wall_s": spec_elapsed,
            "mean_speedup_baseline_over_spec": mean_speedup,
            "mean_alpha": mean_alpha,
            "mean_verify_steps": mean_verify_steps,
            "slower_samples": slower_count,
            "n_samples": int(len(df_pairs)),
            "spec_share_draft": (total_draft / spec_denom),
            "spec_share_verify": (total_verify / spec_denom),
            "spec_share_other": (total_other / spec_denom),
            "net_extra_vs_baseline_s": total_extra,
            "mean_draft_elapsed_s": float(df_pairs["draft_elapsed_s"].mean()) if not df_pairs.empty else float("nan"),
            "mean_verify_elapsed_s": float(df_pairs["verify_elapsed_s"].mean()) if not df_pairs.empty else float("nan"),
            "mean_spec_latency_s": float(df_pairs["spec_latency_s"].mean()) if not df_pairs.empty else float("nan"),
            "mean_baseline_latency_s": float(df_pairs["baseline_latency_s"].mean()) if not df_pairs.empty else float("nan"),
            "mean_timing_remainder_s": mean_timing_remainder,
        }, df_pairs

    except Exception as e:
        debug_meta["exception_type"] = type(e).__name__
        debug_meta["exception_message"] = str(e)
        debug_meta["traceback"] = traceback.format_exc()

        print(f"ERROR in profile {pname}: {type(e).__name__}: {e}")
        print("Traceback:")
        print(debug_meta["traceback"])

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        suffix = f"{quick_draft_label}_{pname}_k{QUICK_K}_tok{QUICK_MAX_NEW_TOKENS}_{draft_device.replace(':','')}_{'adapt' if QUICK_ADAPTIVE_K else 'fixed'}"
        debug_csv = RESULTS_DIR / f"quick_fast_debug_metrics_10samples_{suffix}.csv"
        pd.DataFrame([debug_meta]).to_csv(debug_csv, index=False)
        print(f"Saved debug CSV -> {debug_csv}")

        return {
            "profile": pname,
            "status": "error",
            "error": f"{type(e).__name__}: {e}",
            "target_quant_req": tq,
            "draft_quant_req": dq,
            "target_quant_resolved": _resolved_quant(tq),
            "draft_quant_resolved": _resolved_quant(dq),
            "target_dtype": "",
            "draft_dtype": "",
            "target_load_s": float("nan"),
            "draft_load_s": float("nan"),
            "base_wall_s": float("nan"),
            "spec_wall_s": float("nan"),
            "mean_speedup_baseline_over_spec": float("nan"),
            "mean_alpha": float("nan"),
            "mean_verify_steps": float("nan"),
            "slower_samples": float("nan"),
            "n_samples": 0,
            "spec_share_draft": float("nan"),
            "spec_share_verify": float("nan"),
            "spec_share_other": float("nan"),
            "net_extra_vs_baseline_s": float("nan"),
            "mean_draft_elapsed_s": float("nan"),
            "mean_verify_elapsed_s": float("nan"),
            "mean_spec_latency_s": float("nan"),
            "mean_baseline_latency_s": float("nan"),
            "mean_timing_remainder_s": float("nan"),
        }, df_pairs

    finally:
        _safe_cleanup_models(target_model, draft_model)

all_profile_rows = []
all_pairs_frames = []
for prof in QUICK_QUANT_PROFILES:
    summary_row, df_pairs = run_profile(prof)
    all_profile_rows.append(summary_row)
    if df_pairs is not None and not df_pairs.empty:
        all_pairs_frames.append(df_pairs)

df_profiles = pd.DataFrame(all_profile_rows)
summary_csv = RESULTS_DIR / f"quick_fast_quant_profile_summary_10samples_{quick_draft_label}_k{QUICK_K}_tok{QUICK_MAX_NEW_TOKENS}.csv"
df_profiles.to_csv(summary_csv, index=False)

print("\n" + "=" * 96)
print("Quantization profile summary (higher speedup is better)")
print("=" * 96)
print(df_profiles[[
    "profile", "status", "error",
    "target_quant_req", "target_quant_resolved", "target_dtype",
    "draft_quant_req", "draft_quant_resolved", "draft_dtype",
    "target_load_s", "draft_load_s",
    "mean_speedup_baseline_over_spec",
    "mean_baseline_latency_s", "mean_spec_latency_s",
    "mean_draft_elapsed_s", "mean_verify_elapsed_s",
    "mean_alpha", "mean_verify_steps",
    "slower_samples", "n_samples",
    "mean_timing_remainder_s",
    "net_extra_vs_baseline_s",
]].to_string(index=False))
print(f"Saved summary CSV -> {summary_csv}")

# Cross-profile diagnostics report.
report_lines = []
report_lines.append("=" * 96)
report_lines.append("QUICK QUANT DEBUG REPORT")
report_lines.append("=" * 96)
report_lines.append(f"Timestamp: {datetime.now().isoformat(timespec='seconds')}")
report_lines.append(f"CUDA devices: {torch.cuda.device_count()}")
report_lines.append(
    f"Settings: k={QUICK_K}, max_new_tokens={QUICK_MAX_NEW_TOKENS}, temp={QUICK_TEMP}, top_p={QUICK_TOP_P}, adaptive_k={QUICK_ADAPTIVE_K}"
)
report_lines.append(
    f"Placement: target_device={target_device}, draft_device={draft_device}, colocate={QUICK_COLOCATE_DRAFT_WITH_TARGET}"
)
report_lines.append("")

# Candidate issues checklist.
issues = []

ok_profiles = df_profiles[df_profiles["status"] == "ok"].copy()
if len(ok_profiles) >= 2:
    ref = ok_profiles.iloc[0]
    for _, row in ok_profiles.iloc[1:].iterrows():
        # Fairness check: baseline itself should not drift heavily unless target precision changed.
        if pd.notna(ref["mean_baseline_latency_s"]) and pd.notna(row["mean_baseline_latency_s"]):
            baseline_ratio = float(row["mean_baseline_latency_s"]) / max(float(ref["mean_baseline_latency_s"]), 1e-9)
            if baseline_ratio > 1.5:
                issues.append(
                    f"[Fairness] Baseline latency changed significantly across profiles: {row['profile']} vs {ref['profile']} ratio={baseline_ratio:.2f}x. Cross-profile speedup is influenced by target precision, not only speculative mechanics."
                )

        # Draft regression check.
        if pd.notna(ref["mean_draft_elapsed_s"]) and pd.notna(row["mean_draft_elapsed_s"]):
            draft_ratio = float(row["mean_draft_elapsed_s"]) / max(float(ref["mean_draft_elapsed_s"]), 1e-9)
            if draft_ratio > 1.25:
                issues.append(
                    f"[Draft regression] Mean draft elapsed increased in {row['profile']} vs {ref['profile']} by {draft_ratio:.2f}x."
                )

        # Verify dominance check.
        if pd.notna(row["spec_share_verify"]) and float(row["spec_share_verify"]) > 0.70:
            issues.append(
                f"[Verify bottleneck] {row['profile']} verify share is {100.0*float(row['spec_share_verify']):.1f}% (>70%). Target forward likely dominates runtime."
            )

        # Timing consistency check.
        if pd.notna(row["mean_timing_remainder_s"]) and abs(float(row["mean_timing_remainder_s"])) > 0.02:
            issues.append(
                f"[Timing consistency] {row['profile']} has noticeable component-vs-total remainder mean={float(row['mean_timing_remainder_s']):.4f}s. Consider synchronized per-component timing if strict attribution is required."
            )

        # Quality/acceptance stability check for deterministic runs.
        if pd.notna(ref["mean_alpha"]) and pd.notna(row["mean_alpha"]):
            alpha_gap = abs(float(row["mean_alpha"]) - float(ref["mean_alpha"]))
            if alpha_gap > 0.08:
                issues.append(
                    f"[Behavior drift] Mean alpha differs by {alpha_gap:.3f} between {row['profile']} and {ref['profile']} in deterministic mode. Check sampling flags and seed determinism."
                )

# Sample-level instability check.
if all_pairs_frames:
    df_pairs_all = pd.concat(all_pairs_frames, ignore_index=True)
    pivot = df_pairs_all.pivot_table(index="sample_id", columns="profile", values="extra_vs_baseline_s")
    if pivot.shape[1] >= 2:
        profile_cols = list(pivot.columns)
        a, b = profile_cols[0], profile_cols[1]
        sign_flip = int(((pivot[a] > 0) != (pivot[b] > 0)).sum())
        if sign_flip > 0:
            issues.append(
                f"[Sample sensitivity] {sign_flip}/{len(pivot)} samples flip slow/fast sign between {a} and {b}."
            )

if not issues:
    issues.append("No major anomalies detected by automatic checks.")

report_lines.append("Potential issues detected:")
for i, item in enumerate(issues, start=1):
    report_lines.append(f"{i}. {item}")
report_lines.append("")

report_lines.append("Per-profile key metrics:")
for _, row in df_profiles.iterrows():
    report_lines.append(
        "- "
        f"{row['profile']} | status={row['status']} | speedup={row['mean_speedup_baseline_over_spec']:.4f}x "
        f"| base={row['mean_baseline_latency_s']:.4f}s spec={row['mean_spec_latency_s']:.4f}s "
        f"| draft={row['mean_draft_elapsed_s']:.4f}s verify={row['mean_verify_elapsed_s']:.4f}s "
        f"| alpha={row['mean_alpha']:.4f} | slower={row['slower_samples']}/{int(row['n_samples']) if pd.notna(row['n_samples']) else 0}"
    )

report_text = "\n".join(report_lines)
print("\n" + report_text)

report_path = RESULTS_DIR / f"quick_fast_debug_report_10samples_{quick_draft_label}_k{QUICK_K}_tok{QUICK_MAX_NEW_TOKENS}.txt"
report_path.write_text(report_text, encoding="utf-8")
print(f"\nSaved debug report -> {report_path}")

df_ok = df_profiles[df_profiles["status"] == "ok"].copy()
if not df_ok.empty:
    df_ok = df_ok.sort_values("mean_speedup_baseline_over_spec", ascending=False).reset_index(drop=True)
    best = df_ok.iloc[0]
    print("\nRecommended profile for next runs:")
    print(
        f"  {best['profile']} "
        f"(speedup={best['mean_speedup_baseline_over_spec']:.4f}x, "
        f"slower={int(best['slower_samples'])}/{int(best['n_samples'])})"
    )
else:
    print("\nNo successful profile in this run; inspect the error column above.")


Profile: fp16_target_fp16_draft
Target quant request/resolved: fp16/fp16 | Draft quant request/resolved: fp16/fp16
Config: n=10 k=4 max_new_tokens=12 draft_profile=0.5B colocate=True adaptive_k=False
Loading target on cuda:0 -> Qwen/Qwen2.5-3B-Instruct
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading draft on cuda:0 -> Qwen/Qwen2.5-0.5B-Instruct
Loading model: Qwen/Qwen2.5-0.5B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Model load diagnostics:
  target_load_s=5.49s dtype=torch.bfloat16 alloc/reserved=4258/7152MB
  draft_load_s=0.63s dtype=torch.float16 alloc/reserved=4258/7152MB


base int8_target_fp16_draft:   0%|          | 0/10 [00:00<?, ?it/s]

c:\ProgramData\anaconda3\envs\py12-dl\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


spec int8_target_fp16_draft:   0%|          | 0/10 [00:00<?, ?it/s]

Baseline wall time:    18.34s
Speculative wall time: 10.91s
Mean speedup (baseline/spec): 1.7268x
Mean alpha: 0.4898 | mean verify steps: 4.70
Where speculative runtime is spent: draft=23.0% verify=76.0% other=1.1%
Net extra vs baseline (sum): -7.4286s | slower samples: 0/10
Timing remainder mean (spec - draft - verify - other): 0.000000s
Saved pair CSV -> C:\Working\speculative-decoding-main\results\quick_fast_pair_10samples_0.5B_quick_int8_target_fp16_draft_k4_tok12_cuda0_fixed.csv
Saved breakdown CSV -> C:\Working\speculative-decoding-main\results\quick_fast_breakdown_10samples_0.5B_quick_int8_target_fp16_draft_k4_tok12_cuda0_fixed.csv
Saved debug CSV -> C:\Working\speculative-decoding-main\results\quick_fast_debug_metrics_10samples_0.5B_quick_int8_target_fp16_draft_k4_tok12_cuda0_fixed.csv
Top-3 slowest samples:
  sample_id  extra_vs_baseline_s  alpha  n_verify_steps  draft_elapsed_s  verify_elapsed_s  other_elapsed_s
fast_en_008              -0.3265 0.3333             6.0         

In [2]:
# --- Extended quantization diagnostics: standalone, no dependency on previous cells ---
import os
import sys
import time
import gc
import traceback
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

# Windows/console robustness.
os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("PYTHONLEGACYWINDOWSSTDIO", "1")
os.environ.setdefault("BITSANDBYTES_NOWELCOME", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Offline-first for local cache workflow.
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src" / "runtime.py").exists()), cwd)
SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from config import TARGET_MODEL_ID, DRAFT_MODELS, RESULTS_DIR
from baseline import run_baseline_sample
from speculative import load_model_on_device, speculative_decode_sample
from quantization import get_quant_kwargs

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this extended quick test")

# Tunables (same defaults as quick quant cell).
QUICK_K = 4
QUICK_MAX_NEW_TOKENS = 12
QUICK_TEMP = 0.0
QUICK_TOP_P = 1.0
QUICK_ADAPTIVE_K = False
QUICK_DRAFT_PROFILE = "0.5B"  # "0.5B" or "3B"
QUICK_COLOCATE_DRAFT_WITH_TARGET = True

if QUICK_DRAFT_PROFILE == "3B":
    quick_draft_model_id = TARGET_MODEL_ID
    quick_draft_label = "3B_quick"
else:
    quick_draft_model_id = DRAFT_MODELS["0.5B"]
    quick_draft_label = "0.5B_quick"

target_device = "cuda:0"
if QUICK_COLOCATE_DRAFT_WITH_TARGET or torch.cuda.device_count() < 2:
    draft_device = "cuda:0"
else:
    draft_device = "cuda:1"

quick_samples = [
    {"sample_id": "fast_en_001", "task": "quick_en", "prompt": "Explain in one sentence why leaves are green."},
    {"sample_id": "fast_en_002", "task": "quick_en", "prompt": "Give two tips for improving concentration while studying."},
    {"sample_id": "fast_en_003", "task": "quick_en", "prompt": "Define cloud computing in plain language."},
    {"sample_id": "fast_en_004", "task": "quick_en", "prompt": "Name three benefits of regular exercise."},
    {"sample_id": "fast_en_005", "task": "quick_en", "prompt": "Briefly explain what an API is."},
    {"sample_id": "fast_en_006", "task": "quick_en", "prompt": "Compare supervised and unsupervised learning in two lines."},
    {"sample_id": "fast_en_007", "task": "quick_en", "prompt": "What is overfitting in machine learning?"},
    {"sample_id": "fast_en_008", "task": "quick_en", "prompt": "List two ways to reduce energy consumption at home."},
    {"sample_id": "fast_en_009", "task": "quick_en", "prompt": "Explain the main idea of version control."},
    {"sample_id": "fast_en_010", "task": "quick_en", "prompt": "Write one short tip for clear technical writing."},
]

def _resolved_quant(mode):
    _, resolved = get_quant_kwargs(mode)
    return resolved

def _cuda_mem_mb(device: str) -> dict:
    if not torch.cuda.is_available() or not str(device).startswith("cuda"):
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0, "max_reserved_mb": 0.0}
    idx = int(str(device).split(":")[1])
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
        "max_reserved_mb": torch.cuda.max_memory_reserved(idx) / (1024 ** 2),
    }

def _safe_dtype_of_model(model):
    try:
        return str(next(model.parameters()).dtype)
    except Exception:
        return "unknown"

def _safe_cleanup_models(target_model=None, draft_model=None):
    try:
        if draft_model is not None:
            del draft_model
    except Exception:
        pass
    try:
        if target_model is not None:
            del target_model
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def run_profile(profile: dict) -> tuple[dict, pd.DataFrame]:
    pname = profile["name"]
    tq = profile["target_quant"]
    dq = profile["draft_quant"]

    target_model = None
    draft_model = None

    debug_meta = {
        "profile": pname,
        "target_quant_req": tq,
        "draft_quant_req": dq,
        "target_quant_resolved": _resolved_quant(tq),
        "draft_quant_resolved": _resolved_quant(dq),
        "target_load_s": float("nan"),
        "draft_load_s": float("nan"),
        "target_dtype": "",
        "draft_dtype": "",
        "target_device": target_device,
        "draft_device": draft_device,
        "cuda_count": int(torch.cuda.device_count()),
        "exception_type": "",
        "exception_message": "",
        "traceback": "",
    }

    df_pairs = pd.DataFrame()

    try:
        print("\n" + "=" * 96)
        print(f"Profile: {pname}")
        print(
            f"Target quant request/resolved: {tq}/{_resolved_quant(tq)} | "
            f"Draft quant request/resolved: {dq}/{_resolved_quant(dq)}"
        )
        print(
            f"Config: n={len(quick_samples)} k={QUICK_K} max_new_tokens={QUICK_MAX_NEW_TOKENS} "
            f"draft_profile={QUICK_DRAFT_PROFILE} colocate={QUICK_COLOCATE_DRAFT_WITH_TARGET} adaptive_k={QUICK_ADAPTIVE_K}"
        )

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        print(f"Loading target on {target_device} -> {TARGET_MODEL_ID}")
        t_load_0 = time.perf_counter()
        target_model, target_tokenizer = load_model_on_device(TARGET_MODEL_ID, target_device, tq)
        debug_meta["target_load_s"] = time.perf_counter() - t_load_0
        debug_meta["target_dtype"] = _safe_dtype_of_model(target_model)

        print(f"Loading draft on {draft_device} -> {quick_draft_model_id}")
        d_load_0 = time.perf_counter()
        draft_model, _draft_tokenizer = load_model_on_device(quick_draft_model_id, draft_device, dq)
        debug_meta["draft_load_s"] = time.perf_counter() - d_load_0
        debug_meta["draft_dtype"] = _safe_dtype_of_model(draft_model)

        m_target_after_load = _cuda_mem_mb(target_device)
        m_draft_after_load = _cuda_mem_mb(draft_device)

        print("Model load diagnostics:")
        print(
            f"  target_load_s={debug_meta['target_load_s']:.2f}s dtype={debug_meta['target_dtype']} "
            f"alloc/reserved={m_target_after_load['alloc_mb']:.0f}/{m_target_after_load['reserved_mb']:.0f}MB"
        )
        print(
            f"  draft_load_s={debug_meta['draft_load_s']:.2f}s dtype={debug_meta['draft_dtype']} "
            f"alloc/reserved={m_draft_after_load['alloc_mb']:.0f}/{m_draft_after_load['reserved_mb']:.0f}MB"
        )

        baseline_rows = []
        t0 = time.perf_counter()
        bar_base = tqdm(total=len(quick_samples), desc=f"base {pname}", dynamic_ncols=True, leave=True)
        for i, sample in enumerate(quick_samples, start=1):
            out = run_baseline_sample(
                target_model,
                target_tokenizer,
                sample["prompt"],
                QUICK_MAX_NEW_TOKENS,
                QUICK_TEMP,
                QUICK_TOP_P,
            )
            baseline_rows.append({
                "sample_id": sample["sample_id"],
                "task": sample["task"],
                "mode": "baseline",
                "k": 0,
                "regime": "deterministic",
                **out,
            })
            bar_base.update(1)
            bar_base.set_postfix_str(f"{i}/{len(quick_samples)}")
        bar_base.close()
        base_elapsed = time.perf_counter() - t0

        spec_rows = []
        t1 = time.perf_counter()
        bar_spec = tqdm(total=len(quick_samples), desc=f"spec {pname}", dynamic_ncols=True, leave=True)
        for i, sample in enumerate(quick_samples, start=1):
            out = speculative_decode_sample(
                target_model,
                draft_model,
                target_tokenizer,
                sample["prompt"],
                QUICK_MAX_NEW_TOKENS,
                QUICK_K,
                QUICK_TEMP,
                QUICK_TOP_P,
                return_timing_breakdown=True,
                adaptive_k=QUICK_ADAPTIVE_K,
            )
            spec_rows.append({
                "sample_id": sample["sample_id"],
                "task": sample["task"],
                "mode": "speculative",
                "draft": quick_draft_label,
                "k": QUICK_K,
                "regime": "deterministic",
                "target_device": target_device,
                "draft_device": draft_device,
                "adaptive_k": QUICK_ADAPTIVE_K,
                "profile": pname,
                **{k: v for k, v in out.items() if k != "verify_log"},
            })
            bar_spec.update(1)
            bar_spec.set_postfix_str(f"{i}/{len(quick_samples)}")
        bar_spec.close()
        spec_elapsed = time.perf_counter() - t1

        m_target_after_run = _cuda_mem_mb(target_device)
        m_draft_after_run = _cuda_mem_mb(draft_device)

        base_by_id = {r["sample_id"]: r for r in baseline_rows}
        pairs = []
        for r in spec_rows:
            b = base_by_id.get(r["sample_id"])
            if not b:
                continue

            b_lat = float(b.get("latency_s", 0.0))
            s_lat = float(r.get("latency_s", 0.0))
            draft_t = float(r.get("draft_elapsed_s", 0.0))
            verify_t = float(r.get("verify_elapsed_s", 0.0))
            other_t = max(s_lat - draft_t - verify_t, 0.0)
            timing_gap = s_lat - (draft_t + verify_t + other_t)
            delta_t = s_lat - b_lat
            spec_denom = max(s_lat, 1e-9)

            pairs.append({
                "sample_id": r["sample_id"],
                "profile": pname,
                "baseline_latency_s": b_lat,
                "spec_latency_s": s_lat,
                "extra_vs_baseline_s": delta_t,
                "is_slower_than_baseline": int(delta_t > 0.0),
                "speedup_baseline_over_spec": (b_lat / s_lat) if s_lat > 0 else 0.0,
                "alpha": float(r.get("alpha", 0.0)),
                "B_eff": float(r.get("B_eff", 0.0)),
                "n_verify_steps": float(r.get("n_verify_steps", 0.0)),
                "draft_elapsed_s": draft_t,
                "verify_elapsed_s": verify_t,
                "other_elapsed_s": other_t,
                "timing_remainder_s": timing_gap,
                "draft_share_of_spec": draft_t / spec_denom,
                "verify_share_of_spec": verify_t / spec_denom,
                "other_share_of_spec": other_t / spec_denom,
            })

        df_pairs = pd.DataFrame(pairs)

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        suffix = f"{quick_draft_label}_{pname}_k{QUICK_K}_tok{QUICK_MAX_NEW_TOKENS}_{draft_device.replace(':','')}_{'adapt' if QUICK_ADAPTIVE_K else 'fixed'}"
        base_csv = RESULTS_DIR / f"quick_fast_baseline_10samples_{suffix}.csv"
        spec_csv = RESULTS_DIR / f"quick_fast_spec_10samples_{suffix}.csv"
        pair_csv = RESULTS_DIR / f"quick_fast_pair_10samples_{suffix}.csv"
        breakdown_csv = RESULTS_DIR / f"quick_fast_breakdown_10samples_{suffix}.csv"
        debug_csv = RESULTS_DIR / f"quick_fast_debug_metrics_10samples_{suffix}.csv"

        pd.DataFrame(baseline_rows).to_csv(base_csv, index=False)
        pd.DataFrame(spec_rows).to_csv(spec_csv, index=False)
        if not df_pairs.empty:
            df_pairs.to_csv(pair_csv, index=False)
            df_pairs.to_csv(breakdown_csv, index=False)

        debug_rows = [{
            **debug_meta,
            "target_alloc_mb_after_load": m_target_after_load["alloc_mb"],
            "target_reserved_mb_after_load": m_target_after_load["reserved_mb"],
            "target_peak_alloc_mb_after_run": m_target_after_run["max_alloc_mb"],
            "target_peak_reserved_mb_after_run": m_target_after_run["max_reserved_mb"],
            "draft_alloc_mb_after_load": m_draft_after_load["alloc_mb"],
            "draft_reserved_mb_after_load": m_draft_after_load["reserved_mb"],
            "draft_peak_alloc_mb_after_run": m_draft_after_run["max_alloc_mb"],
            "draft_peak_reserved_mb_after_run": m_draft_after_run["max_reserved_mb"],
            "base_wall_s": base_elapsed,
            "spec_wall_s": spec_elapsed,
            "n_samples": int(len(df_pairs)),
        }]
        pd.DataFrame(debug_rows).to_csv(debug_csv, index=False)

        total_spec = float(df_pairs["spec_latency_s"].sum()) if not df_pairs.empty else 0.0
        total_draft = float(df_pairs["draft_elapsed_s"].sum()) if not df_pairs.empty else 0.0
        total_verify = float(df_pairs["verify_elapsed_s"].sum()) if not df_pairs.empty else 0.0
        total_other = float(df_pairs["other_elapsed_s"].sum()) if not df_pairs.empty else 0.0
        total_extra = float(df_pairs["extra_vs_baseline_s"].sum()) if not df_pairs.empty else 0.0
        spec_denom = max(total_spec, 1e-9)

        mean_speedup = float(df_pairs["speedup_baseline_over_spec"].mean()) if not df_pairs.empty else 0.0
        mean_alpha = float(df_pairs["alpha"].mean()) if not df_pairs.empty else 0.0
        mean_verify_steps = float(df_pairs["n_verify_steps"].mean()) if not df_pairs.empty else 0.0
        slower_count = int(df_pairs["is_slower_than_baseline"].sum()) if not df_pairs.empty else 0

        return {
            "profile": pname,
            "status": "ok",
            "error": "",
            "target_quant_req": tq,
            "draft_quant_req": dq,
            "target_quant_resolved": _resolved_quant(tq),
            "draft_quant_resolved": _resolved_quant(dq),
            "target_dtype": debug_meta["target_dtype"],
            "draft_dtype": debug_meta["draft_dtype"],
            "target_load_s": debug_meta["target_load_s"],
            "draft_load_s": debug_meta["draft_load_s"],
            "base_wall_s": base_elapsed,
            "spec_wall_s": spec_elapsed,
            "mean_speedup_baseline_over_spec": mean_speedup,
            "mean_alpha": mean_alpha,
            "mean_verify_steps": mean_verify_steps,
            "slower_samples": slower_count,
            "n_samples": int(len(df_pairs)),
            "spec_share_draft": (total_draft / spec_denom),
            "spec_share_verify": (total_verify / spec_denom),
            "spec_share_other": (total_other / spec_denom),
            "net_extra_vs_baseline_s": total_extra,
            "mean_draft_elapsed_s": float(df_pairs["draft_elapsed_s"].mean()) if not df_pairs.empty else float("nan"),
            "mean_verify_elapsed_s": float(df_pairs["verify_elapsed_s"].mean()) if not df_pairs.empty else float("nan"),
            "mean_spec_latency_s": float(df_pairs["spec_latency_s"].mean()) if not df_pairs.empty else float("nan"),
            "mean_baseline_latency_s": float(df_pairs["baseline_latency_s"].mean()) if not df_pairs.empty else float("nan"),
        }, df_pairs

    except Exception as e:
        debug_meta["exception_type"] = type(e).__name__
        debug_meta["exception_message"] = str(e)
        debug_meta["traceback"] = traceback.format_exc()

        print(f"ERROR in profile {pname}: {type(e).__name__}: {e}")
        print("Traceback:")
        print(debug_meta["traceback"])

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        suffix = f"{quick_draft_label}_{pname}_k{QUICK_K}_tok{QUICK_MAX_NEW_TOKENS}_{draft_device.replace(':','')}_{'adapt' if QUICK_ADAPTIVE_K else 'fixed'}"
        debug_csv = RESULTS_DIR / f"quick_fast_debug_metrics_10samples_{suffix}.csv"
        pd.DataFrame([debug_meta]).to_csv(debug_csv, index=False)

        return {
            "profile": pname,
            "status": "error",
            "error": f"{type(e).__name__}: {e}",
            "target_quant_req": tq,
            "draft_quant_req": dq,
            "target_quant_resolved": _resolved_quant(tq),
            "draft_quant_resolved": _resolved_quant(dq),
            "target_dtype": "",
            "draft_dtype": "",
            "target_load_s": float("nan"),
            "draft_load_s": float("nan"),
            "base_wall_s": float("nan"),
            "spec_wall_s": float("nan"),
            "mean_speedup_baseline_over_spec": float("nan"),
            "mean_alpha": float("nan"),
            "mean_verify_steps": float("nan"),
            "slower_samples": float("nan"),
            "n_samples": 0,
            "spec_share_draft": float("nan"),
            "spec_share_verify": float("nan"),
            "spec_share_other": float("nan"),
            "net_extra_vs_baseline_s": float("nan"),
            "mean_draft_elapsed_s": float("nan"),
            "mean_verify_elapsed_s": float("nan"),
            "mean_spec_latency_s": float("nan"),
            "mean_baseline_latency_s": float("nan"),
        }, df_pairs

    finally:
        _safe_cleanup_models(target_model, draft_model)

EXTENDED_QUANT_PROFILES = [
    {"name": "fp16_target_fp16_draft", "target_quant": "fp16", "draft_quant": "fp16"},
    {"name": "int8_target_fp16_draft", "target_quant": "int8", "draft_quant": "fp16"},
    {"name": "int8_target_int8_draft", "target_quant": "int8", "draft_quant": "int8"},
]

ext_rows = []
ext_pairs = []
for prof in EXTENDED_QUANT_PROFILES:
    row, pairs = run_profile(prof)
    ext_rows.append(row)
    if pairs is not None and not pairs.empty:
        ext_pairs.append(pairs)

df_profiles_ext = pd.DataFrame(ext_rows)

# Two speed definitions.
df_profiles_ext["decode_only_speedup"] = (
    df_profiles_ext["mean_baseline_latency_s"]
    / df_profiles_ext["mean_spec_latency_s"].clip(lower=1e-9)
)
df_profiles_ext["baseline_total_with_target_load_s"] = (
    df_profiles_ext["base_wall_s"] + df_profiles_ext["target_load_s"]
)
df_profiles_ext["spec_total_with_all_load_s"] = (
    df_profiles_ext["spec_wall_s"] + df_profiles_ext["target_load_s"] + df_profiles_ext["draft_load_s"]
)
df_profiles_ext["e2e_with_load_speedup"] = (
    df_profiles_ext["baseline_total_with_target_load_s"]
    / df_profiles_ext["spec_total_with_all_load_s"].clip(lower=1e-9)
)

df_profiles_ext["target_quant_fallback"] = (
    df_profiles_ext["target_quant_req"] != df_profiles_ext["target_quant_resolved"]
)
df_profiles_ext["draft_quant_fallback"] = (
    df_profiles_ext["draft_quant_req"] != df_profiles_ext["draft_quant_resolved"]
)

suffix = f"{quick_draft_label}_k{QUICK_K}_tok{QUICK_MAX_NEW_TOKENS}"
ext_summary_csv = RESULTS_DIR / f"quick_fast_quant_profile_summary_extended_10samples_{suffix}.csv"
df_profiles_ext.to_csv(ext_summary_csv, index=False)

print("\n" + "=" * 104)
print("EXTENDED QUANTIZATION SUMMARY (decode-only + end-to-end-with-load)")
print("=" * 104)
print(df_profiles_ext[[
    "profile", "status", "error",
    "target_quant_req", "target_quant_resolved", "target_quant_fallback",
    "draft_quant_req", "draft_quant_resolved", "draft_quant_fallback",
    "target_dtype", "draft_dtype",
    "mean_baseline_latency_s", "mean_spec_latency_s",
    "decode_only_speedup",
    "base_wall_s", "spec_wall_s",
    "target_load_s", "draft_load_s",
    "baseline_total_with_target_load_s", "spec_total_with_all_load_s",
    "e2e_with_load_speedup",
    "mean_draft_elapsed_s", "mean_verify_elapsed_s",
    "mean_alpha", "mean_verify_steps", "slower_samples", "n_samples",
]].to_string(index=False))
print(f"Saved extended summary CSV -> {ext_summary_csv}")

report_lines = []
report_lines.append("=" * 104)
report_lines.append("EXTENDED DEBUG REPORT")
report_lines.append("=" * 104)
report_lines.append(f"Timestamp: {datetime.now().isoformat(timespec='seconds')}")

ok_ext = df_profiles_ext[df_profiles_ext["status"] == "ok"].copy()
if ok_ext.empty:
    report_lines.append("No successful profiles in extended run.")
else:
    report_lines.append("\nBest profile by decode-only speedup:")
    best_decode = ok_ext.sort_values("decode_only_speedup", ascending=False).iloc[0]
    report_lines.append(
        f"- {best_decode['profile']} (decode_only_speedup={best_decode['decode_only_speedup']:.4f}x)"
    )

    report_lines.append("\nBest profile by end-to-end-with-load speedup:")
    best_e2e = ok_ext.sort_values("e2e_with_load_speedup", ascending=False).iloc[0]
    report_lines.append(
        f"- {best_e2e['profile']} (e2e_with_load_speedup={best_e2e['e2e_with_load_speedup']:.4f}x)"
    )

    report_lines.append("\nFixed-target draft-only comparisons:")
    for target_mode, grp in ok_ext.groupby("target_quant_resolved"):
        grp2 = grp.sort_values("decode_only_speedup", ascending=False)
        report_lines.append(f"- target={target_mode}:")
        for _, r in grp2.iterrows():
            report_lines.append(
                f"  {r['profile']} | draft={r['draft_quant_resolved']} | "
                f"decode_only={r['decode_only_speedup']:.4f}x | e2e_with_load={r['e2e_with_load_speedup']:.4f}x"
            )

    report_lines.append("\nPotential caveats:")
    fallback_rows = ok_ext[(ok_ext["target_quant_fallback"]) | (ok_ext["draft_quant_fallback"])]
    if not fallback_rows.empty:
        report_lines.append(
            f"- Quantization fallback detected in {len(fallback_rows)} profile(s): requested mode not honored."
        )
    high_verify = ok_ext[ok_ext["mean_verify_elapsed_s"] > ok_ext["mean_draft_elapsed_s"] * 2.0]
    if not high_verify.empty:
        report_lines.append(
            f"- Verify-dominant runtime in {len(high_verify)} profile(s); target forward path is the primary bottleneck."
        )
    if fallback_rows.empty and high_verify.empty:
        report_lines.append("- No additional caveats detected.")

ext_report_text = "\n".join(report_lines)
print("\n" + ext_report_text)

ext_report_path = RESULTS_DIR / f"quick_fast_debug_report_extended_10samples_{suffix}.txt"
ext_report_path.write_text(ext_report_text, encoding="utf-8")
print(f"\nSaved extended debug report -> {ext_report_path}")


Profile: fp16_target_fp16_draft
Target quant request/resolved: fp16/fp16 | Draft quant request/resolved: fp16/fp16
Config: n=10 k=4 max_new_tokens=12 draft_profile=0.5B colocate=True adaptive_k=False
Loading target on cuda:0 -> Qwen/Qwen2.5-3B-Instruct
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading draft on cuda:0 -> Qwen/Qwen2.5-0.5B-Instruct
Loading model: Qwen/Qwen2.5-0.5B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Model load diagnostics:
  target_load_s=4.83s dtype=torch.bfloat16 alloc/reserved=3908/4620MB
  draft_load_s=0.98s dtype=torch.bfloat16 alloc/reserved=3908/4620MB


base int8_target_int8_draft:   0%|          | 0/10 [00:00<?, ?it/s]

c:\ProgramData\anaconda3\envs\py12-dl\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


spec int8_target_int8_draft:   0%|          | 0/10 [00:00<?, ?it/s]


EXTENDED QUANTIZATION SUMMARY (decode-only + end-to-end-with-load)
               profile status error target_quant_req target_quant_resolved  target_quant_fallback draft_quant_req draft_quant_resolved  draft_quant_fallback   target_dtype    draft_dtype  mean_baseline_latency_s  mean_spec_latency_s  decode_only_speedup  base_wall_s  spec_wall_s  target_load_s  draft_load_s  baseline_total_with_target_load_s  spec_total_with_all_load_s  e2e_with_load_speedup  mean_draft_elapsed_s  mean_verify_elapsed_s  mean_alpha  mean_verify_steps  slower_samples  n_samples
fp16_target_fp16_draft     ok                   fp16                  fp16                  False            fp16                 fp16                 False  torch.float16  torch.float16                  0.30196              0.36961             0.816969     3.040177     3.715825       2.632616      0.667515                           5.672793                    7.015957               0.808556              0.250557               0.1

## Phase 3 — Speculative Grid A: Qwen2.5-0.5B Draft

Run speculative decoding with 0.5B draft across k ∈ {4, 8, 16} × {deterministic, stochastic} = 6 configs.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
from runtime import bootstrap_notebook, ensure_data, ensure_baseline_results, ensure_spec_results
from speculative import run_speculative_grid, load_model_on_device
from config import TARGET_MODEL_ID, TARGET_QUANT, DRAFT_QUANT, DRAFT_MODELS

bootstrap_notebook()
if "install_notebook_progress_helpers" in globals():
    install_notebook_progress_helpers()

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Phase 3 requires at least 2 CUDA devices: target on cuda:0, draft on cuda:1")

data = ensure_data(globals())
ensure_baseline_results(globals())
spec_results_05 = ensure_spec_results(globals(), "0.5B")

# Pin target to cuda:0 and 0.5B draft to cuda:1.
target_model, target_tokenizer = load_model_on_device(
    TARGET_MODEL_ID,
    "cuda:0",
    TARGET_QUANT,
 )
draft_05_model, draft_05_tokenizer = load_model_on_device(
    DRAFT_MODELS["0.5B"],
    "cuda:1",
    DRAFT_QUANT,
 )

loaded_target = str(getattr(target_model, "name_or_path", ""))
loaded_draft = str(getattr(draft_05_model, "name_or_path", ""))
print(f"Configured target model: {TARGET_MODEL_ID}")
print(f"Loaded target model: {loaded_target} on {target_model.device}")
print(f"Loaded draft model: {loaded_draft} on {next(draft_05_model.parameters()).device}")
assert loaded_target == TARGET_MODEL_ID, f"Target mismatch: loaded={loaded_target}, expected={TARGET_MODEL_ID}"
assert target_tokenizer.vocab_size == draft_05_tokenizer.vocab_size, "Tokenizer vocab mismatch between target and 0.5B draft"
print("Phase 3 device split active: target->cuda:0, draft->cuda:1")


  [gsm8k] loaded 300 samples from manifest
  [mmlu] loaded 500 samples from manifest
  [cnndm] loaded 200 samples from manifest
Loading target model: Qwen/Qwen2.5-3B-Instruct (quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  gsm8k (exact_match): 59.00%  (n=300)
  mmlu (letter_match): 56.60%  (n=500)
  cnndm (rouge_l): 20.90%  (n=200)
  gsm8k (exact_match): 56.33%  (n=300)
  mmlu (letter_match): 56.80%  (n=500)
  cnndm (rouge_l): 20.38%  (n=200)
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: Qwen/Qwen2.5-0.5B-Instruct (device=cuda:1, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Configured target model: Qwen/Qwen2.5-3B-Instruct
Loaded target model: Qwen/Qwen2.5-3B-Instruct on cuda:0
Loaded draft model: Qwen/Qwen2.5-0.5B-Instruct on cuda:1
Phase 3 device split active: target->cuda:0, draft->cuda:1


In [2]:
# --- Quick debug run (first 200 samples) ---
from config import (
    REGIMES, RESULTS_DIR, TARGET_MODEL_ID, TARGET_QUANT, DRAFT_QUANT, DRAFT_MODELS,
    )
from utils import write_csv
from speculative import run_speculative_grid, load_model_on_device
from runtime import bootstrap_notebook, ensure_data

bootstrap_notebook()

# Bind runner locally so this cell can run independently.
spec_grid_runner = run_speculative_grid

# Ensure required objects exist even when this cell is run standalone.
data = ensure_data(globals())
if "target_model" not in globals() or "target_tokenizer" not in globals():
    target_model, target_tokenizer = load_model_on_device(
        TARGET_MODEL_ID,
        "cuda:0",
        TARGET_QUANT,
    )
if "draft_05_model" not in globals() or "draft_05_tokenizer" not in globals():
    draft_05_model, draft_05_tokenizer = load_model_on_device(
        DRAFT_MODELS["0.5B"],
        "cuda:1",
        DRAFT_QUANT,
    )

# Keep debug run small and isolated from full-grid outputs.
debug_max_samples = 200
debug_k = 4
debug_draft_label = "0.5B_debug200"
debug_regimes = ("deterministic", "stochastic")

remaining = debug_max_samples
data_debug_05 = {}
for task_name, samples in data.items():
    if remaining <= 0:
        break
    take_n = min(len(samples), remaining)
    if take_n > 0:
        data_debug_05[task_name] = samples[:take_n]
        remaining -= take_n

debug_total = sum(len(v) for v in data_debug_05.values())
debug_breakdown = {k: len(v) for k, v in data_debug_05.items()}
print(f"Quick debug subset: {debug_total} samples")
print(f"Subset breakdown: {debug_breakdown}")

debug_results_05 = globals().get("debug_results_05", {})

for regime_name in debug_regimes:
    config_key = f"{debug_draft_label}_k{debug_k}_{regime_name}"
    print(f"{'=' * 60}")
    print(f"DEBUG Speculative: draft={debug_draft_label}, k={debug_k}, regime={regime_name}, samples={debug_total}")
    print(f"{'=' * 60}")

    results = spec_grid_runner(
        data_debug_05, "0.5B", debug_k, regime_name,
        target_model, target_tokenizer,
        draft_05_model, draft_05_tokenizer,
        show_realtime_progress=True,
        target_device_label="cuda:0",
        draft_device_label="cuda:1",
    )

    debug_results_05[config_key] = results

    # Persist a dedicated debug CSV so this run never overwrites full-grid artifacts.
    regime_short = "det" if regime_name == "deterministic" else "stoch"
    debug_csv_path = RESULTS_DIR / f"spec_0.5B_k{debug_k}_{regime_short}_debug200.csv"
    write_csv(debug_csv_path, [{key: val for key, val in r.items() if key != "verify_log"} for r in results])
    print(f"Saved debug CSV -> {debug_csv_path}")

print(f"Quick debug run complete: {list(debug_results_05.keys())}")

  [gsm8k] loaded 300 samples from manifest
  [mmlu] loaded 500 samples from manifest
  [cnndm] loaded 200 samples from manifest
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: Qwen/Qwen2.5-0.5B-Instruct (device=cuda:1, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

target verify cuda:0:   0%|          | 0/200 [00:00<?, ?it/s]

  Saved -> C:\Working\speculative-decoding-main\results\spec_0.5B_k4_det.csv
  Verify logs -> C:\Working\speculative-decoding-main\results\verify_logs\spec_0.5B_k4_det_seed42.json
Saved debug CSV -> C:\Working\speculative-decoding-main\results\spec_0.5B_k4_det_debug200.csv
DEBUG Speculative: draft=0.5B_debug200, k=4, regime=stochastic, samples=200


draft 0.5B cuda:1:   0%|          | 0/200 [00:00<?, ?it/s]

target verify cuda:0:   0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [2]:
from config import DRAFT_LENGTHS, REGIMES

spec_results_05 = globals().get("spec_results_05", {})
spec_grid_runner = run_speculative_grid

# Set True to rerun even if CSV-backed results already exist.
force_rerun_phase3 = False

# Use the full dataset (1000 samples) for 0.5B grid runs.
data_full_05 = data
total_samples_05 = sum(len(v) for v in data_full_05.values())
breakdown_05 = {k: len(v) for k, v in data_full_05.items()}
print(f"Using full dataset for 0.5B grid: {total_samples_05} samples")
print(f"Dataset breakdown: {breakdown_05}")
print("Realtime dual bars enabled: draft(cuda:1) + target verify(cuda:0)")

for k_val in DRAFT_LENGTHS:
    for regime_name in REGIMES:
        config_key = f"0.5B_k{k_val}_{regime_name}"
        if (not force_rerun_phase3) and config_key in spec_results_05 and spec_results_05[config_key]:
            print(f"Skipping existing config: {config_key}")
            continue

        print(f"{'=' * 60}")
        print(f"Speculative: draft=0.5B, k={k_val}, regime={regime_name}, samples={total_samples_05}")
        print(f"{'=' * 60}")

        results = spec_grid_runner(
            data_full_05, "0.5B", k_val, regime_name,
            target_model, target_tokenizer,
            draft_05_model, draft_05_tokenizer,
            show_realtime_progress=True,
            target_device_label="cuda:0",
            draft_device_label="cuda:1",
        )
        spec_results_05[config_key] = results

print(f"Grid A ready: {len(spec_results_05)} configs")


Using full dataset for 0.5B grid: 1000 samples
Dataset breakdown: {'gsm8k': 300, 'mmlu': 500, 'cnndm': 200}
Realtime dual bars enabled: draft(cuda:1) + target verify(cuda:0)
Speculative: draft=0.5B, k=4, regime=deterministic, samples=1000


draft 0.5B cuda:1:   0%|          | 0/1000 [00:00<?, ?it/s]

target verify cuda:0:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# --- Grid A: Evaluate quality and compute metrics ---
from metrics import compute_acceptance_metrics, compute_speedup, compute_quality_delta, compute_latency_metrics
from evaluate import evaluate_results

grid_a_summary = []
for config_key, results in spec_results_05.items():
    parts = config_key.split("_")  # e.g. '0.5B_k4_deterministic'
    k_val = int(parts[1][1:])
    regime = parts[2]

    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    base_quality_ref = base_quality_det if regime == "deterministic" else base_quality_stoch

    lat = compute_latency_metrics(results)
    acc = compute_acceptance_metrics(results)
    quality = evaluate_results(results, data)
    speedup = compute_speedup(baseline_ref, results)
    delta_q = compute_quality_delta(base_quality_ref, quality)

    print(f"\n{config_key}: S={speedup:.2f}x, α={acc['alpha']:.3f}, B_eff={acc['B_eff']:.1f}")
    for task_k, dq in delta_q.items():
        print(f"  {task_k}: {dq:+.2f}pp")

    grid_a_summary.append({
        "config": config_key, "draft": "0.5B", "k": k_val, "regime": regime,
        **lat, **acc, "S": speedup, **quality, **delta_q,
    })

df_grid_a = pd.DataFrame(grid_a_summary)
df_grid_a

## Phase 4 — Speculative Grid B: Qwen2.5-1.5B Draft

Run speculative decoding with 1.5B draft across the same 6 configs, plus stability analysis.

In [ ]:
import os
import sys
from pathlib import Path
import torch

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import gc
from runtime import bootstrap_notebook, ensure_data, ensure_baseline_results, ensure_spec_results
from speculative import run_speculative_grid, load_model_on_device
from config import TARGET_MODEL_ID, TARGET_QUANT, DRAFT_QUANT, DRAFT_MODELS

bootstrap_notebook()
if "install_notebook_progress_helpers" in globals():
    install_notebook_progress_helpers()

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Phase 4 requires at least 2 CUDA devices: target on cuda:0, draft on cuda:1")

data = ensure_data(globals())
ensure_baseline_results(globals())
spec_results_05 = ensure_spec_results(globals(), "0.5B")
spec_results_15 = ensure_spec_results(globals(), "1.5B")

# Keep target pinned on cuda:0.
target_model, target_tokenizer = load_model_on_device(
    TARGET_MODEL_ID,
    "cuda:0",
    TARGET_QUANT,
 )

# Free 0.5B draft to save memory, then load 1.5B draft on cuda:1.
if "draft_05_model" in globals():
    del draft_05_model
if "draft_05_tokenizer" in globals():
    del draft_05_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

draft_15_model, draft_15_tokenizer = load_model_on_device(
    DRAFT_MODELS["1.5B"],
    "cuda:1",
    DRAFT_QUANT,
 )

loaded_target = str(getattr(target_model, "name_or_path", ""))
loaded_draft = str(getattr(draft_15_model, "name_or_path", ""))
print(f"Configured target model: {TARGET_MODEL_ID}")
print(f"Loaded target model: {loaded_target} on {target_model.device}")
print(f"Loaded draft model: {loaded_draft} on {next(draft_15_model.parameters()).device}")
assert loaded_target == TARGET_MODEL_ID, f"Target mismatch: loaded={loaded_target}, expected={TARGET_MODEL_ID}"
assert target_tokenizer.vocab_size == draft_15_tokenizer.vocab_size, "Tokenizer vocab mismatch between target and 1.5B draft"
print("Phase 4 device split active: target->cuda:0, draft->cuda:1")


In [ ]:
from config import DRAFT_LENGTHS, REGIMES

spec_results_15 = globals().get("spec_results_15", {})
spec_grid_runner = run_speculative_grid

# Set True to rerun even if CSV-backed results already exist.
force_rerun_phase4 = False

# Use the full dataset (1000 samples) for 1.5B grid runs.
data_full_15 = data
total_samples_15 = sum(len(v) for v in data_full_15.values())
breakdown_15 = {k: len(v) for k, v in data_full_15.items()}
print(f"Using full dataset for 1.5B grid: {total_samples_15} samples")
print(f"Dataset breakdown: {breakdown_15}")
print("Realtime dual bars enabled: draft(cuda:1) + target verify(cuda:0)")

for k_val in DRAFT_LENGTHS:
    for regime_name in REGIMES:
        config_key = f"1.5B_k{k_val}_{regime_name}"
        if (not force_rerun_phase4) and config_key in spec_results_15 and spec_results_15[config_key]:
            print(f"Skipping existing config: {config_key}")
            continue

        print(f"{'=' * 60}")
        print(f"Speculative: draft=1.5B, k={k_val}, regime={regime_name}, samples={total_samples_15}")
        print(f"{'=' * 60}")

        results = spec_grid_runner(
            data_full_15, "1.5B", k_val, regime_name,
            target_model, target_tokenizer,
            draft_15_model, draft_15_tokenizer,
            show_realtime_progress=True,
            target_device_label="cuda:0",
            draft_device_label="cuda:1",
        )
        spec_results_15[config_key] = results

print(f"Grid B ready: {len(spec_results_15)} configs")


In [ ]:
# --- Grid B: Evaluate quality and compute metrics ---
grid_b_summary = []
for config_key, results in spec_results_15.items():
    parts = config_key.split("_")
    k_val = int(parts[1][1:])
    regime = parts[2]

    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    base_quality_ref = base_quality_det if regime == "deterministic" else base_quality_stoch

    lat = compute_latency_metrics(results)
    acc = compute_acceptance_metrics(results)
    quality = evaluate_results(results, data)
    speedup = compute_speedup(baseline_ref, results)
    delta_q = compute_quality_delta(base_quality_ref, quality)

    print(f"\n{config_key}: S={speedup:.2f}x, α={acc['alpha']:.3f}, B_eff={acc['B_eff']:.1f}")
    for task_k, dq in delta_q.items():
        print(f"  {task_k}: {dq:+.2f}pp")

    grid_b_summary.append({
        "config": config_key, "draft": "1.5B", "k": k_val, "regime": regime,
        **lat, **acc, "S": speedup, **quality, **delta_q,
    })

df_grid_b = pd.DataFrame(grid_b_summary)
df_grid_b

In [ ]:
# --- Combine all 12 configs into master table ---
df_all = pd.concat([df_grid_a, df_grid_b], ignore_index=True)
print("\nFull 12-config results matrix:")
display(df_all[["config", "draft", "k", "regime", "S", "alpha", "B_eff",
                "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms"]])

# Save master table
df_all.to_csv(RESULTS_DIR / "all_configs_summary.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'all_configs_summary.csv'}")

### Phase 4b — Stability Analysis (Top 2 Configs)

Identify the top-2 configs by speedup (subject to |ΔQ| ≤ 1.0) and re-run with seeds {42, 123, 999}.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from runtime import bootstrap_notebook, ensure_data, ensure_baseline_results, ensure_spec_results, ensure_df_all
from metrics import compute_speedup_stability
from speculative import run_stability_analysis

bootstrap_notebook()

data = ensure_data(globals())
ensure_baseline_results(globals())
spec_results_05 = ensure_spec_results(globals(), "0.5B")
spec_results_15 = ensure_spec_results(globals(), "1.5B")
if "df_all" not in globals() or df_all.empty:
    df_all = ensure_df_all(globals())


In [ ]:
# Run stability analysis for top 2 configs
stability_results = {}

for _, row in top2.iterrows():
    draft_label = row["draft"]
    k_val = int(row["k"])
    regime = row["regime"]
    config_key = row["config"]

    print(f"\n{'=' * 60}")
    print(f"Stability analysis: {config_key}")
    print(f"{'=' * 60}")

    # Load correct draft model
    if draft_label == "0.5B":
        dm, dt = load_draft_model("0.5B")
    else:
        dm, dt = draft_15_model, draft_15_tokenizer

    seed_runs = run_stability_analysis(
        data, draft_label, k_val, regime,
        target_model, target_tokenizer, dm, dt,
    )

    # Compute speedup per seed
    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    speedups = []
    for sr in seed_runs:
        s = compute_speedup(baseline_ref, sr["results"])
        speedups.append(s)
        print(f"  seed={sr['seed']}: S={s:.4f}")

    sigma = compute_speedup_stability(speedups)
    print(f"  σ_S = {sigma:.4f}")
    stability_results[config_key] = {"speedups": speedups, "sigma_S": sigma}

print("\n✓ Stability analysis complete")

## Phase 5 — Quality and Error Analysis

Systematic analysis of disagreement patterns between speculative and baseline outputs, including length buckets, taxonomy, and rejection behavior proxies.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from difflib import SequenceMatcher
import numpy as np
from evaluate import cnndm_rouge_l
from runtime import bootstrap_notebook, ensure_data, ensure_baseline_results, ensure_spec_results

bootstrap_notebook()

data = ensure_data(globals())
ensure_baseline_results(globals())
spec_results_05 = ensure_spec_results(globals(), "0.5B")
spec_results_15 = ensure_spec_results(globals(), "1.5B")

# Merge all speculative runs
all_spec_results = {**spec_results_05, **spec_results_15}

# Baseline lookup by regime + sample_id
baseline_lookup = {
    "deterministic": {r["sample_id"]: r for r in baseline_det},
    "stochastic": {r["sample_id"]: r for r in baseline_stoch},
}

rows = []
for config_name, cfg_rows in all_spec_results.items():
    regime = cfg_rows[0]["regime"] if cfg_rows else "deterministic"
    for r in cfg_rows:
        sid = r["sample_id"]
        base = baseline_lookup[regime].get(sid)
        if base is None:
            continue

        pred_text = (r.get("output_text") or "").strip()
        base_text = (base.get("output_text") or "").strip()

        pred_words = len(pred_text.split())
        base_words = len(base_text.split())
        disagree = pred_text != base_text
        sim_ratio = SequenceMatcher(None, pred_text, base_text).ratio()

        if pred_words <= 16:
            length_bucket = "short"
        elif pred_words <= 64:
            length_bucket = "medium"
        else:
            length_bucket = "long"

        rouge_to_base = np.nan
        if r["task"] == "cnndm" and pred_text and base_text:
            rouge_to_base = cnndm_rouge_l(pred_text, base_text)

        rows.append({
            "config": config_name,
            "draft": r["draft"],
            "k": r["k"],
            "regime": regime,
            "task": r["task"],
            "sample_id": sid,
            "disagree": disagree,
            "pred_words": pred_words,
            "base_words": base_words,
            "length_bucket": length_bucket,
            "sim_ratio": sim_ratio,
            "rouge_to_base": rouge_to_base,
            "alpha_sample": (r.get("total_accepted", 0) / r.get("total_proposed", 1)) if r.get("total_proposed", 0) > 0 else np.nan,
        })

df_phase5 = pd.DataFrame(rows)
print(f"Constructed Phase 5 analysis table: {len(df_phase5)} rows")
display(df_phase5.head(10))


In [ ]:
# Disagreement rates by config/task and by output length bucket

df_disagree = df_phase5[df_phase5["disagree"]].copy()

disagree_rate = (
    df_phase5.groupby(["config", "task"], as_index=False)["disagree"]
    .mean()
    .rename(columns={"disagree": "disagreement_rate"})
)

bucket_dist = (
    df_disagree.groupby(["config", "length_bucket"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not bucket_dist.empty:
    bucket_dist["pct_within_config"] = (
        bucket_dist["count"]
        / bucket_dist.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

print("Disagreement rate by config/task:")
display(disagree_rate.sort_values(["disagreement_rate", "config"], ascending=[False, True]).head(20))

print("\nLength bucket distribution among disagreement cases:")
display(bucket_dist.sort_values(["config", "length_bucket"]))

In [ ]:
# Error taxonomy + rejection clustering proxy

def classify_error(row):
    if not row["disagree"]:
        return "match"

    pred_w = max(row["pred_words"], 1)
    base_w = max(row["base_words"], 1)

    # Truncation: output substantially shorter than baseline output.
    if pred_w < 0.6 * base_w:
        return "truncation"

    # Semantic drift: low lexical similarity and (for CNNDM) low ROUGE-L to baseline output.
    if row["sim_ratio"] < 0.25:
        return "semantic_drift"
    if row["task"] == "cnndm" and not np.isnan(row["rouge_to_base"]) and row["rouge_to_base"] < 0.2:
        return "semantic_drift"

    return "substitution"


df_phase5["error_type"] = df_phase5.apply(classify_error, axis=1)

taxonomy = (
    df_phase5[df_phase5["error_type"] != "match"]
    .groupby(["config", "error_type"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not taxonomy.empty:
    taxonomy["pct_within_config"] = (
        taxonomy["count"]
        / taxonomy.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

# Rejection-position clustering requires verify-step position logs.
# Current CSV rows do not keep verify_log, so we provide an acceptance-ratio proxy.
proxy = df_phase5.dropna(subset=["alpha_sample"]).copy()
proxy["rejection_proxy"] = pd.cut(
    proxy["alpha_sample"],
    bins=[-1, 0.33, 0.66, 1.0],
    labels=["early_reject_proxy", "mid_reject_proxy", "late_reject_proxy"],
)
rej_proxy_table = (
    proxy.groupby(["config", "rejection_proxy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not rej_proxy_table.empty:
    rej_proxy_table["pct_within_config"] = (
        rej_proxy_table["count"]
        / rej_proxy_table.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

has_verify_logs = any("verify_log" in row for cfg in all_spec_results.values() for row in cfg)
print(f"verify_log persisted in run outputs: {has_verify_logs}")
if not has_verify_logs:
    print("NOTE: true token-position rejection clustering is not possible from current saved rows; using acceptance-ratio proxy instead.")

print("\nError taxonomy among disagreement cases:")
display(taxonomy.sort_values(["config", "count"], ascending=[True, False]))

print("\nRejection clustering proxy:")
display(rej_proxy_table.sort_values(["config", "rejection_proxy"]))

## Phase 6 — Synthesis and Visualization

Generate Pareto/acceptance figures, synthesis tables, and an execution-plan audit for missing items.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import matplotlib.pyplot as plt
import seaborn as sns
from runtime import bootstrap_notebook, ensure_df_all

bootstrap_notebook()

if "df_all" not in globals() or df_all.empty:
    df_all = ensure_df_all(globals())
if "stability_results" not in globals():
    stability_results = {}

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

df_plot = df_all.copy()
delta_cols = [c for c in df_plot.columns if c.startswith("delta_")]
if not delta_cols:
    delta_cols = [c for c in df_plot.columns if c.startswith("dQ_")]

df_plot["max_abs_delta"] = df_plot[delta_cols].abs().max(axis=1) if delta_cols else 0.0

# Pareto scatter: Speedup vs max quality drift
plt.figure(figsize=(8, 5))
ax = sns.scatterplot(
    data=df_plot,
    x="S",
    y="max_abs_delta",
    hue="draft",
    style="regime",
    s=90,
)
ax.set_title("Pareto: Speedup vs Max |Delta Q|")
ax.set_xlabel("Speedup S (x)")
ax.set_ylabel("Max |Delta Q| (pp)")
plt.tight_layout()
pareto_path = FIGURES_DIR / "pareto_speedup_vs_quality_delta.png"
plt.savefig(pareto_path, dpi=160)
plt.show()

# Acceptance vs k grouped by draft and regime
alpha_k = (
    df_plot.groupby(["draft", "regime", "k"], as_index=False)["alpha"]
    .mean()
)

plt.figure(figsize=(8, 5))
ax = sns.lineplot(data=alpha_k, x="k", y="alpha", hue="draft", style="regime", marker="o")
ax.set_title("Acceptance Rate alpha vs Draft Length k")
ax.set_xlabel("k")
ax.set_ylabel("alpha")
plt.tight_layout()
alpha_k_path = FIGURES_DIR / "acceptance_vs_k.png"
plt.savefig(alpha_k_path, dpi=160)
plt.show()

# Acceptance by regime (temperature proxy)
alpha_reg = (
    df_plot.groupby(["regime", "draft"], as_index=False)["alpha"]
    .mean()
)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=alpha_reg, x="regime", y="alpha", hue="draft")
ax.set_title("Acceptance Rate alpha by Regime")
ax.set_xlabel("Regime")
ax.set_ylabel("alpha")
plt.tight_layout()
alpha_reg_path = FIGURES_DIR / "acceptance_by_regime.png"
plt.savefig(alpha_reg_path, dpi=160)
plt.show()

print("Saved figures:")
print(f"  - {pareto_path}")
print(f"  - {alpha_k_path}")
print(f"  - {alpha_reg_path}")


In [ ]:
# Synthesis tables for manuscript

results_matrix_cols = [
    "config", "draft", "k", "regime", "S", "alpha", "B_eff",
    "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms",
    "gsm8k", "mmlu", "cnndm",
]
results_matrix_cols = [c for c in results_matrix_cols if c in df_all.columns]

results_matrix = df_all[results_matrix_cols].sort_values(["draft", "k", "regime"]).copy()
display(results_matrix)
results_matrix.to_csv(RESULTS_DIR / "table_full_results_matrix.csv", index=False)

# Best speculative config per task + baseline comparison
quality_table_rows = []
baseline_quality_map = {
    "deterministic": base_quality_det,
    "stochastic": base_quality_stoch,
}

for task in ["gsm8k", "mmlu", "cnndm"]:
    if task not in df_all.columns:
        continue
    best_idx = df_all[task].idxmax()
    best_row = df_all.loc[best_idx]
    regime = best_row["regime"]
    baseline_score = baseline_quality_map[regime][task]
    quality_table_rows.append({
        "task": task,
        "baseline_regime": regime,
        "baseline_score": baseline_score,
        "best_spec_config": best_row["config"],
        "best_spec_score": best_row[task],
        "delta_spec_minus_base": round(best_row[task] - baseline_score, 2),
    })

df_quality_compare = pd.DataFrame(quality_table_rows)
print("Best speculative config per task (vs matching baseline regime):")
display(df_quality_compare)
df_quality_compare.to_csv(RESULTS_DIR / "table_quality_comparison.csv", index=False)

print("Saved tables:")
print(f"  - {RESULTS_DIR / 'table_full_results_matrix.csv'}")
print(f"  - {RESULTS_DIR / 'table_quality_comparison.csv'}")

In [ ]:
# Audit checklist: what is still missing for Phase 6 paper completion
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "sec").exists():
    # Fallback to notebook path root behavior if executed from elsewhere
    project_root = Path(SRC_DIR).parent

checks = []

# 1) Missing sections requested by execution plan
for rel in ["sec/01b_related_work.tex", "sec/06_results.tex"]:
    p = project_root / rel
    checks.append({
        "item": f"Section file exists: {rel}",
        "status": "PASS" if p.exists() else "MISSING",
        "evidence": str(p),
    })

# 2) Abstract placeholders still present?
abstract_path = project_root / "sec/00_abstract.tex"
if abstract_path.exists():
    abstract_text = abstract_path.read_text(encoding="utf-8", errors="ignore")
    placeholders_present = any(tok in abstract_text for tok in ["[X]", "[Y]", "[Z]"])
    checks.append({
        "item": "Abstract placeholders resolved ([X],[Y],[Z])",
        "status": "MISSING" if placeholders_present else "PASS",
        "evidence": "Placeholders found" if placeholders_present else "No placeholders found",
    })

# 3) Rejection-position instrumentation availability
has_verify_logs = any("verify_log" in row for cfg in all_spec_results.values() for row in cfg)
checks.append({
    "item": "Token-position rejection logs persisted",
    "status": "PASS" if has_verify_logs else "MISSING",
    "evidence": "verify_log present in result rows" if has_verify_logs else "verify_log dropped in run_speculative_grid output rows",
})

# 4) Visualization module implementation
viz_path = project_root / "src/visualize.py"
if viz_path.exists():
    viz_text = viz_path.read_text(encoding="utf-8", errors="ignore")
    stub = "implemented later" in viz_text.lower()
    checks.append({
        "item": "src/visualize.py implemented",
        "status": "MISSING" if stub else "PASS",
        "evidence": "Stub marker found" if stub else "No stub marker found",
    })

# 5) Ensure phase outputs written
required_outputs = [
    project_root / "results/all_configs_summary.csv",
    project_root / "results/all_configs_combined.csv",
    project_root / "results/table_full_results_matrix.csv",
    project_root / "results/table_quality_comparison.csv",
]
for out in required_outputs:
    checks.append({
        "item": f"Output generated: {out.name}",
        "status": "PASS" if out.exists() else "PENDING",
        "evidence": str(out),
    })

df_audit = pd.DataFrame(checks)
print("Phase 6 audit report:")
display(df_audit)

missing_items = df_audit[df_audit["status"].isin(["MISSING", "PENDING"])]
if missing_items.empty:
    print("\nAll checklist items passed.")
else:
    print("\nItems still pending/missing:")
    display(missing_items)

## Phase 7 — DriftDiffuse: Parallel Diffusion Drafter

Phases 2–6 close out the **autoregressive (AR)** speculative-decoding analysis: we
quantified accept rate, $B_{\text{eff}}$, throughput, and quality across
$\{0.5\text{B}, 1.5\text{B}\}$ drafts and three values of $k$. The empirical
$B_{\text{eff}}$ tracks the geometric ceiling $\frac{1-\alpha^{k+1}}{1-\alpha}$
within ~3%, so the binding term left on the table is the **linear-in-$k$ draft cost**
$T_{\text{draft}} = k \cdot t_{\text{draft}}$.

Phase 7 evaluates **DriftDiffuse**, our purpose-built ~30M-parameter
**bidirectional masked-diffusion drafter**, on the same target. Instead of
unrolling $k$ AR steps, the drafter:

1. Initialises a $k$-token block as all `[MASK]`.
2. Runs $n_{\text{denoise}}$ parallel forward passes (typically $n_{\text{denoise}} \in \{2, 3\} \ll k$).
3. Reveals positions earlier in the block first via a position-conditional
   **drift schedule**, so left-to-right plausibility is preserved while keeping the
   bidirectional context window intact.
4. Feeds the proposed block to the same target verifier, with **block accept**
   (Bernoulli on the geometric mean of per-token ratios) and a Leviathan
   token-level rejection sampler as fallback so the output distribution is
   provably preserved.

Because $n_{\text{denoise}}$ is constant in $k$, the draft cost stops scaling
with $k$, which is exactly the wall-clock term that bottlenecks the AR sweep at
$k{\geq}8$.

![DriftDiffuse pipeline](figures/driftdiffuse_pipeline.png)

> **Caveat for the 3B target.** As discussed in `docs/target_model_choice.md`,
> the verifier-amortisation budget on a 3B target is small. We expect
> DriftDiffuse to be most competitive at $n_{\text{denoise}}{\leq}3$ with the
> deterministic regime; results below quantify exactly where it falls relative
> to the AR best (Phase 4).


In [ ]:
# --- Phase 7: setup (free AR draft memory, load DriftDiffuse drafter) ---
import os
import sys
import gc
from pathlib import Path

import torch

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src").exists()), None)
assert project_root is not None, "Could not locate project root containing src/"
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from runtime import bootstrap_notebook, ensure_target_model, ensure_drifter

ns = bootstrap_notebook(globals())

# Free the 1.5B AR draft if it is still resident — we only need the target
# verifier and the small drifter from here on.
for name in ("draft_15_model", "draft_15_tokenizer", "draft_05_model", "draft_05_tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Make sure the verifier (Qwen2.5-3B-Instruct, int8) is loaded.
target_model, target_tokenizer = ensure_target_model(globals())

# Load the trained DriftDiffuse drafter from results/drifter_ckpt/drifter_latest.pt.
# If the checkpoint is missing, set train_if_missing=True (uses src/diffusion/train.py).
drifter, drift_schedule = ensure_drifter(globals(), train_if_missing=False)

n_params = drifter.num_params
print(f"DriftDiffuser loaded — {n_params/1e6:.1f}M params")
print(f"  hidden={drifter.cfg.hidden}, layers={drifter.cfg.n_layers}, "
      f"heads={drifter.cfg.n_heads}, k_max={drifter.cfg.k_max}, "
      f"max_ctx_len={drifter.cfg.max_ctx_len}")
print(f"  vocab_size={drifter.cfg.vocab_size} (matches target tokenizer: "
      f"{drifter.cfg.vocab_size == len(target_tokenizer)})")


In [ ]:
# --- Phase 7: run DriftDiffuse on the GSM8K eval subset ---
# Small grid so the run completes in the same wall-clock budget as one AR config.
# n_denoise ∈ {2, 3} keeps the parallel-drafter advantage intact on a 3B verifier.
from runtime import ensure_drift_results

drift_results = ensure_drift_results(
    globals(),
    k_values=(8, 16),
    n_denoise_steps=(2, 3),
    accept_modes=("block",),
    regimes=("deterministic",),
    label="drift",
)

print(f"DriftDiffuse configs run: {len(drift_results)}")
for key, rows in drift_results.items():
    print(f"  {key}: {len(rows)} samples")


In [ ]:
# --- Phase 7: metrics + head-to-head comparison vs AR best ---
import pandas as pd

from metrics import (
    compute_acceptance_metrics,
    compute_latency_metrics,
    compute_speedup,
    compute_quality_delta,
)
from evaluate import evaluate_results
from config import RESULTS_DIR

# Baseline reference (loaded by ensure_drift_results -> ensure_baseline_results).
baseline_results = globals().get("baseline_results", {})
baseline_det_rows = baseline_results.get("deterministic", [])

drift_summary = []
for key, rows in drift_results.items():
    # key format: "{label}_n{n_denoise}_{accept_mode}_k{k}_{regime}"
    parts = key.split("_")
    n_denoise = int(parts[1].lstrip("n"))
    accept_mode = parts[2]
    k = int(parts[3].lstrip("k"))
    regime = parts[4]

    accept = compute_acceptance_metrics(rows)
    latency = compute_latency_metrics(rows)
    quality = evaluate_results(rows)

    base_lat = compute_latency_metrics(baseline_det_rows) if baseline_det_rows else {}
    base_qual = evaluate_results(baseline_det_rows) if baseline_det_rows else {}
    speedup = compute_speedup(latency, base_lat) if base_lat else {}
    delta_q = compute_quality_delta(quality, base_qual) if base_qual else {}

    drift_summary.append({
        "config": f"DriftDiffuse-n{n_denoise}-{accept_mode}",
        "draft": "DriftDiffuse",
        "k": k,
        "regime": regime,
        "n_denoise": n_denoise,
        "accept_mode": accept_mode,
        "alpha": accept.get("alpha"),
        "B_eff": accept.get("B_eff"),
        "tokens_per_sec": latency.get("tokens_per_sec"),
        "speedup": speedup.get("throughput_speedup"),
        **{f"quality_{k_}": v for k_, v in quality.items()},
        **{f"delta_{k_}": v for k_, v in delta_q.items()},
    })

df_drift = pd.DataFrame(drift_summary).sort_values(["k", "n_denoise"]).reset_index(drop=True)
print("DriftDiffuse summary:")
display(df_drift)

# Head-to-head vs AR best (read from df_all if present).
df_all = globals().get("df_all")
if df_all is not None and not df_all.empty:
    ar_det = df_all[df_all["regime"] == "deterministic"].copy()
    ar_best = ar_det.sort_values("speedup", ascending=False).head(3)
    drift_best = df_drift.sort_values("speedup", ascending=False).head(3)
    print("\nTop-3 AR (deterministic):")
    display(ar_best[["config", "draft", "k", "alpha", "B_eff", "tokens_per_sec", "speedup"]])
    print("\nTop-3 DriftDiffuse:")
    display(drift_best[["config", "k", "n_denoise", "alpha", "B_eff", "tokens_per_sec", "speedup"]])

# Persist the merged table for the paper.
df_drift.to_csv(RESULTS_DIR / "drift_summary.csv", index=False)
print(f"\nSaved -> {RESULTS_DIR / 'drift_summary.csv'}")


### Transition: Phase 7 → Summary

Phase 7 closes the experimental campaign. The Summary below stitches together
the AR sweep (Phases 2–4) and the DriftDiffuse sweep (Phase 7) into the final
14+ row results matrix referenced by the paper.


## Summary

Display the final results matrix: 2 baselines + 12 AR speculative + DriftDiffuse sweep, plus stability info from Phase 4b.

In [ ]:
# Final summary table
print("=" * 80)
print("EXPERIMENT COMPLETE — FULL RESULTS MATRIX")
print("=" * 80)

display_cols = [c for c in ["config", "draft", "k", "regime", "S", "alpha", "B_eff",
                "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms"] if c in df_all.columns]

print("\n--- Latency & Throughput ---")
display(df_all[display_cols].sort_values("R_tok_mean", ascending=False))

quality_cols = [c for c in df_all.columns if c in ["gsm8k", "mmlu", "cnndm"]]
if quality_cols:
    print("\n--- Quality (%) ---")
    display(df_all[["config"] + quality_cols])

# Best config under quality constraint
if not df_qualified.empty:
    best = df_qualified.loc[df_qualified["S"].idxmax()]
    print(f"\n★ Best config (|ΔQ| ≤ 1.0): {best['config']}")
    print(f"  Speedup: {best['S']:.2f}x, α: {best['alpha']:.3f}")

# Save combined table
df_all.to_csv(RESULTS_DIR / "all_configs_combined.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'all_configs_combined.csv'}")

# Stability
if stability_results:
    print("\n--- Stability (σ_S) ---")
    for cfg, info in stability_results.items():
        print(f"  {cfg}: σ_S = {info['sigma_S']:.4f}, seeds = {info['speedups']}")